# QEvict completion ablation for Reviewer 2 (Colab, A100)

> *How sensitive is QEvict's model quality to the accuracy of the calibrated window-level
> completion term for windows that are not selected for token-level attention?*

**Model:** Llama-3.1-8B-Instruct. **Operating point:** `utils.config.OPERATING_POINT`: budget 0.20, W = 8, `quant_ratio` 0.70
in `bytes` mode, read ratio ρ = 0.25, int2, flash backend, `bidir`/`dequant`, greedy.

| Experiment | Question | Where |
|---|---|---|
| **1** Matched-state accuracy | How close is the completion mass to the true unread-window mass, and how much output error does each variant cause **on the same decode state**? | collector, `--completion-rungs` |
| **2** Is completion needed? | `cal` vs `off` (and the ρ = 1.0 ceiling) | LongBench |
| **3** Is calibration needed? | `cal` vs `uncal` (`exp(ℓ̂ − log Z_A)`) | LongBench |
| **4** How sensitive is quality? | common scale `c`; total-preserving noise `σ`; coupled scores | LongBench |

`cal` is the shipped method, the control in every experiment. Two floors are measured on 10 examples per task:
* `cal_repeat`, a second run of the same configuration, gives the run-to-run noise;
* `cal_ep` sends the shipped term through the evaluation epilogue at identity, which isolates the epilogue's own numerics.

### The knob (`utils/completion_spec.py`)
* `cache.quant_completion`: `null` runs the shipped kernel path. Any other value (`calibrated`, `off`, `uncalibrated`) runs the
  evaluation epilogue (`modules/windowed_cache/completion.py`). The epilogue recovers `ō`, the fill and `log Z_A` exactly from
  the kernel's own output and applies the variant **after selection and after `δ`**. The kernel source is unchanged.
* `quant_completion_scale`: `c`. `quant_completion_sigma` (in nats) with `quant_completion_seed`: noise that preserves the total,
  drawn per layer × query head × window id and fixed for the window's life. `quant_completion_scores`: `output` leaves the
  eviction scores on the shipped fill; `coupled` moves them too.
* Use `off`, not `none`: the override parser turns `none` into null.

### Why a number can be trusted
Each LongBench sidecar must carry all of the following, or the result is moved to `refused/` and is not scored:
* `read_gate.verdict == "gated"`;
* the realised read fraction for the requested ρ;
* the operating point;
* the requested `num_samples`;
* `quant_completion` equal to the arm's spec;
* runtime proof: `read_gate.completion_applied == read_gate.gated` on variant arms, and `0` on native arms.

One Drive folder holds one commit. **Run every arm on the same commit.** The notebook refuses to start if the knob is missing.

### Run size: an ablation, not a benchmark
| `RUN` | What | One A100 (arithmetic, not measured) |
|---|---|---|
| `"core"` (default) | Exp 1 on 4 tasks × 4 prompts × 128 steps (~0.5M head-steps per task); LongBench at 40 examples/task for `cal`, `off`, `uncal`, `scale2`, `noise1x`; floors at 10/task | **~2–2.5 h** |
| `"extended"` | adds the ρ = 1.0 ceiling, scales 0.5/0.8/1.25/4, more noise seeds and levels, the ρ = 0.10 pair, coupled scores, GSM8K (250) | about +5 h at 40/task |

**The precision comes from Experiment 1.** It scores every variant on the same decode states, at about half a million
head-steps per task: all five scales, all four noise levels and both oracles. That is cheap, and it gives a tight
sensitivity curve. The LongBench arms confirm the key points end to end. At 40 examples per task they detect only
large changes; the power check prints the exact smallest detectable change. `NUM_SAMPLES` scales the time linearly.

The estimates assume ~55 ms/token at B=1 plus prefill, +20% for epilogue arms, and one model load per arm. After the
first arm, the status cell re-projects the rest from the **measured** examples per second.

### How to use
1. Runtime → **A100** (High-RAM helps Experiment 1). Colab secrets: `HF_TOKEN` (Llama-3.1 access), and `GITHUB_TOKEN` if the repo is private.
2. Run top to bottom. Every finished arm is kept on Drive: after a disconnect, run everything again.
   The pinned stack is installed once per runtime into a Python 3.12 venv (`/content/py312_env`);
   section 2 rebuilds it only if it is missing or was not built by this notebook.
3. The original extended grid (`RUN = "extended"`) runs only with `ORIGINAL_FROZEN = False`. For the Reviewer-2
   follow-up points, use Part II (step 4), which runs them without rerunning anything.
4. **Reviewer-2 follow-up (Part II, sections 10–15).** Leave `RUN = "core"` and `ORIGINAL_FROZEN = True`, choose the
   `FU_*` settings, and run everything. Sections 6–9 then only read Drive. Part II runs only jobs whose rows are not
   on Drive yet, and section 11 prints the plan and its projected time first (`FU_RUN_JOBS = False` stops there).

### Part II: the Reviewer-2 follow-up (sections 10–15)
| Section | Adds |
|---|---|
| **12** | ρ = 1.0 full read: an availability check on all four tasks, and the comparison with `cal` at ρ = 0.25 |
| **13** | ρ = 0.10 stress test: `cal` vs `off`, end to end and on matched states (Experiment 1 at ρ = 0.10) |
| **14** | end-to-end sensitivity: c ∈ {0.5, 0.8, 1.25, 4}, and noise at 2× and 4× σ̂_U over seeds 0, 1, 2 |
| **11** | more rows per task for the key arms (`FU_N_KEY`), by running **only the new rows** |
| **15** | one table of attention-output error, accuracy and cost at the fixed KV-memory budget, and a generated summary that keeps the internal-error reduction apart from the downstream deltas |

**Nothing already measured is rewritten or rerun:**
* `ORIGINAL_FROZEN = True` makes sections 6–9 read-only.
* Part II writes only under `DRIVE_ROOT/followup/`.
* Every arm already on Drive is reused at the N it ran at.
* No finished file is ever overwritten.

Section 10 has the details.

## 0. Settings

In [ ]:
# ===================== Settings: edit here =====================
BRANCH       = "Clustered_int2_paused_fast"
REPO_URL     = "https://github.com/shaswata965-coder/RecoverKv.git"
PIN_COMMIT   = "5924f0329b61ca0b43f0131be6201f6f1893220c"   # every arm runs on this commit (see below)
# 5924f03 is the last commit that touched the ablation code. The branch keeps moving (later commits are
# LongGenBench work), and quality rows do not compare across commits, so the run is pinned. Set None
# only together with a new DRIVE_ROOT.
MODEL_ID     = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_DIR    = "/content/models/Llama-3.1-8B-Instruct"   # the name must contain "instruct": the chat-template gate reads it
DRIVE_ROOT   = "/content/drive/MyDrive/recoverkv_completion_ablation"

RUN          = "core"    # "core" (~2-2.5 h) or "extended" (adds the rest; finished arms are kept)
TASKS        = ["narrativeqa", "musique", "qmsum", "triviaqa"]
NUM_SAMPLES        = 40      # LongBench examples per task, every arm; pairs use the first N. Time scales linearly
#                             This is the core run's N: do not raise it for more rows. FU_N_KEY (Part II) adds rows,
#                             and runs only the new ones.
NUM_SAMPLES_FLOOR  = 10      # cal_repeat and cal_ep measure floors, not effects
RUN_GSM8K          = False   # extended only
GSM8K_NUM_SAMPLES  = 250     # the full split is 1,319 CoT answers: about 10 h per arm

# Experiment 1: collector on the shipped method, every variant computed on the same states
EXP1_PROMPTS = 4         # x 128 steps x 32 layers x 32 heads = ~0.5M head-steps per task
EXP1_PREFILL = 4096
EXP1_GEN     = 128
EXP1_CAL_VS_KERNEL_MAX = 1e-2   # refuse Exp 1 if the recomputed calibrated output strays this far from the kernel's (median)

BOOTSTRAP_ITERS        = 10_000
RUN_PREFLIGHT_TESTS    = True
FLASH_ATTN_FROM_SOURCE = False   # only if no prebuilt wheel matches (slow: ~1-2 h)
ALLOW_NON_A100         = False

# ===================== Part II: the Reviewer-2 follow-up (sections 10-15) =====================
ORIGINAL_FROZEN   = True    # sections 6-9 only READ Drive: they launch no job and rewrite no file
FU_RUN_JOBS       = True    # False: section 11 prints the plan and its projected time, and runs nothing
FU_STEPS          = ["ceiling", "stress", "exp1_stress", "scales", "noise", "topup"]   # run order = priority
FU_SCALES         = (0.5, 0.8, 1.25, 4.0)   # c; 2.0 is the core run's scale2
FU_NOISE_MULTS    = (2, 4)                  # sigma = mult x sigma_hat_U; 1x seed 0 is the core run's noise1x_s0
FU_NOISE_SEEDS    = (0, 1, 2)
FU_NOISE_1X_SEEDS = False   # also run 1x at seeds 1 and 2 (not asked for; 2 more arms)
FU_N_SENS         = None    # rows/task for the new arms; None = the N the core cal arm ran at (read from Drive)
FU_N_KEY          = 100     # top-up target in rows/task for FU_TOPUP_ARMS (task files hold 200); None = no top-up
FU_TOPUP_ARMS     = ["cal", "off", "ceiling_g1.0", "stress_g0.10_cal", "stress_g0.10_off"]
PAPER_ERROR_REDUCTION = 0.58   # the matched-state figure quoted to the reviewer; section 15 recomputes and checks it

# ===================== Fixed: these define the experiment =====================
OPERATING_POINT = dict(cache_budget=0.20, window_size=8, quant_ratio=0.70,
                       quant_budget_mode="bytes", quant_promotion="bidir",
                       quant_promote_source="dequant")
GATE = 0.25
STRESS_GATE = 0.10   # the stress arms of section 5 are named for it
STAGES = {"core": 1, "extended": 2}
assert RUN in STAGES, RUN

## 1. GPU, Drive, repository

In [ ]:
import subprocess, json, os, sys, shutil, time, math, glob, zipfile, io, re, warnings, difflib
from pathlib import Path

q = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                    "--format=csv,noheader,nounits"], capture_output=True, text=True)
if q.returncode != 0:
    raise RuntimeError("No GPU. Runtime -> Change runtime type -> A100.")
gpu_name, mem_mib, cc = [x.strip() for x in q.stdout.strip().splitlines()[0].split(",")]
GPU_MEM_GB = float(mem_mib) / 1024
print(f"GPU: {gpu_name}  {GPU_MEM_GB:.0f} GB  sm_{cc.replace('.', '')}")
if cc != "8.0" and not ALLOW_NON_A100:
    raise RuntimeError(f"Expected an A100 (sm_80), got {gpu_name} (sm_{cc}). The kernels were "
                       "compile-checked for sm80. Set ALLOW_NON_A100 = True to override.")
SKIP_OOM = GPU_MEM_GB < 70
print(f"skip_oom = {SKIP_OOM}" + ("  (40 GB card: an OOM example scores 0 THUDM-style and is dropped "
      "from paired comparisons in every arm)" if SKIP_OOM else ""))

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
ROOT = Path(DRIVE_ROOT)
for d in ("longbench", "gsm8k", "exp1", "logs", "refused", "results"):
    (ROOT / d).mkdir(parents=True, exist_ok=True)

def _secret(k):
    try:
        return userdata.get(k)
    except Exception:
        return None

REPO = Path("/content/RecoverKv")
gh = _secret("GITHUB_TOKEN")
url = REPO_URL.replace("https://", f"https://x-access-token:{gh}@") if gh else REPO_URL

def _git(*args, cwd=None):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:  # never echo the tokenised URL
        raise RuntimeError("git " + args[0] + " failed:\n" + r.stderr.replace(gh or "\0", "***"))
    return r.stdout.strip()

if not REPO.exists():
    _git("clone", "--branch", BRANCH, url, str(REPO))
else:
    _git("fetch", "origin", BRANCH, cwd=REPO)
    _git("checkout", "-B", BRANCH, f"origin/{BRANCH}", cwd=REPO)
if PIN_COMMIT:
    _git("checkout", PIN_COMMIT, cwd=REPO)
SHA = _git("rev-parse", "HEAD", cwd=REPO)
print("commit", SHA, "(pinned)" if PIN_COMMIT else "  <- set PIN_COMMIT to this before any rerun")

# Quality rows do not compare across commits, so one Drive root holds one commit.
manifest = ROOT / "manifest.json"
if manifest.exists():
    m = json.loads(manifest.read_text())
    if m["sha"] != SHA:
        raise RuntimeError(f"{ROOT} holds results from commit {m['sha'][:10]}, and this clone is "
                           f"{SHA[:10]}. Set PIN_COMMIT = '{m['sha'][:10]}' to resume, or point "
                           "DRIVE_ROOT at a new folder.")
else:
    manifest.write_text(json.dumps({"sha": SHA, "branch": BRANCH, "model": MODEL_ID,
                                    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())},
                                   indent=2))

## 2. Install the pinned stack

Colab's kernel is Python 3.13 with its own, newer torch. The pinned stack goes into a separate
**Python 3.12 venv** (`/content/py312_env`), and every job below runs with that interpreter (`PY`).
The notebook kernel never imports torch, so no runtime restart is needed.

* torch 2.6.0 is the build the recorded GPU runs used. Do not move it: on torch 2.7.1 the eviction's
  `torch.compile` fails at decode step 0 (`StarDep does not have an index`), which stops every arm.
* transformers is pinned to 4.47.1: the windowed cache refuses anything newer.
* `datasets` stays below 3, as `environment.yml` pins it.
* flash-attn comes from a **prebuilt wheel**, installed with `--no-deps`, and is kept only if its CUDA
  extension actually imports against the venv's torch. In practice that is **2.7.4.post1**: the 2.8.x wheels
  labelled `cu12torch2.6cxx11abiFALSE` are built with the new C++ ABI and fail against pip's torch 2.6.0
  with `undefined symbol: _ZN3c105ErrorC2ENS_14SourceLocationENSt7__cxx11...` (checked: torch 2.6.0 exports
  that constructor only in the old-ABI form).
* Every install after torch runs under a constraints file (`torch==2.6.0`, `triton==3.2.0`, ...), so nothing
  can move torch.

**Why the previous version of this cell failed** (`undefined symbol:
_ZN3c104cuda29c10_cuda_check_implementationEiPKcS2_ib`): it installed flash-attn with `--force-reinstall`.
That flag reinstalls the package's dependencies too, and flash-attn depends on an unpinned `torch`, so pip
replaced torch 2.6.0 with the newest torch on PyPI. The flash-attn binary had been built against torch 2.6's
headers, and the newer libtorch no longer exports that symbol in that form. Had torch stayed at 2.6.0, the
2.8.3 wheel would still have failed, on the ABI symbol above. The cell now pins torch with a constraints
file, checks the installed versions before anything imports flash-attn, and tests the import itself.

If no wheel matches, set `FLASH_ATTN_FROM_SOURCE = True` (about 1–2 h, under the same constraints).

In [ ]:
import importlib.util

VENV = Path("/content/py312_env")
PY = str(VENV / "bin" / "python")          # every job runs with this interpreter
if sys.executable.startswith(str(VENV)):   # an older version of this cell pointed the kernel at the venv
    sys.executable = getattr(sys, "_base_executable", None) or "/usr/bin/python3"
KERNEL_PY = sys.executable                 # the notebook kernel itself: runs only the scoring and plots

PINS = {"torch": "2.6.0", "torchvision": "0.21.0", "torchaudio": "2.6.0", "triton": "3.2.0",
        "transformers": "4.47.1"}
STACK_ID = json.dumps({"python": "3.12", "cuda": "cu124", **PINS}, sort_keys=True)
MARK = VENV / ".recoverkv_stack"           # written only after the environment check below passes

# Nothing from Colab's own Python may leak into the venv, at install time or at run time.
_DROP = ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "CPATH", "C_INCLUDE_PATH", "CPLUS_INCLUDE_PATH",
         "PIP_CONSTRAINT", "MPLBACKEND")
VENV_ENV = {k: v for k, v in os.environ.items() if k not in _DROP}
VENV_ENV.update(VIRTUAL_ENV=str(VENV), PATH=f"{VENV / 'bin'}:{os.environ.get('PATH', '')}",
                MPLBACKEND="Agg")   # scripts that call plain `python` get the venv too

def sh(*cmd, env=None):
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd)[:300])
    subprocess.run(cmd, check=True, env=VENV_ENV if env is None else env)

def vpip(*a):
    sh(PY, "-m", "pip", "install", "-q", "--disable-pip-version-check", *a)

def venv_versions():
    code = ("import json, importlib.metadata as m\nout = {}\n"
            "for d in ('torch','torchvision','torchaudio','triton','transformers','flash_attn','datasets','numpy'):\n"
            "    try: out[d] = m.version(d)\n    except m.PackageNotFoundError: out[d] = None\n"
            "print(json.dumps(out))")
    return json.loads(subprocess.check_output([PY, "-c", code], env=VENV_ENV, text=True))

def check_pins():
    v = venv_versions()
    bad = {k: v.get(k) for k, want in PINS.items() if (v.get(k) or "").split("+")[0] != want}
    if bad:
        raise RuntimeError(f"the venv drifted from the pinned stack: {bad}, want {PINS}. Something "
                           "reinstalled torch (e.g. a --force-reinstall, or an unpinned dependency). "
                           f"Delete {VENV} and rerun this cell.")
    return v

if not (MARK.exists() and MARK.read_text() == STACK_ID and Path(PY).exists()):
    if VENV.exists():
        print(f"removing {VENV}: not built by this cell, or for a different stack")
        shutil.rmtree(VENV)
    has312 = shutil.which("python3.12") and subprocess.run(
        ["python3.12", "-c", "import ensurepip, venv"], capture_output=True).returncode == 0
    if not has312 or FLASH_ATTN_FROM_SOURCE:  # a source build also needs the Python headers
        apt_env = dict(os.environ, DEBIAN_FRONTEND="noninteractive")
        sh("apt-get", "update", "-qq", env=apt_env)
        sh("apt-get", "install", "-y", "-qq", "python3.12", "python3.12-venv", "python3.12-dev", env=apt_env)
    sh("python3.12", "-m", "venv", VENV, env=os.environ)
    constraints = VENV / "constraints.txt"
    constraints.write_text("".join(f"{k}=={v}\n" for k, v in PINS.items()))
    vpip("--upgrade", "pip")
    vpip("torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0",
         "--index-url", "https://download.pytorch.org/whl/cu124")
    vpip("-c", constraints,                 # no bare "triton": torch 2.6.0 brings 3.2.0
         "transformers==4.47.1", "accelerate>=0.25,<2", "datasets>=2.16,<3", "huggingface_hub>=0.24,<1.0",
         "sentencepiece", "protobuf", "einops>=0.7", "scipy", "pyyaml", "tqdm", "pandas", "matplotlib",
         "rouge", "fuzzywuzzy", "python-Levenshtein", "jieba", "pytest")
    check_pins()

    tv, abi, py = subprocess.check_output([PY, "-c",
        "import torch, sys; print(torch.__version__.split('+')[0]); "
        "print(int(torch._C._GLIBCXX_USE_CXX11_ABI)); "
        "print(f'cp{sys.version_info.major}{sys.version_info.minor}')"], env=VENV_ENV, text=True).split()
    tmm = ".".join(tv.split(".")[:2])

    def _fa_loads():
        # The binary must resolve every symbol against THIS torch. Importing the extension is the test;
        # a wheel's file name is not: the 2.8.x "cxx11abiFALSE" wheels for torch 2.6 are built with the
        # new C++ ABI and fail with an undefined c10::Error symbol.
        r = subprocess.run([PY, "-c", "import torch, flash_attn_2_cuda, flash_attn; print(flash_attn.__version__)"],
                           env=VENV_ENV, capture_output=True, text=True)
        return r.returncode == 0, (r.stdout + r.stderr).strip().splitlines()[-1:]

    wheel = None
    for v in ("2.7.4.post1", "2.8.3", "2.8.2"):
        u = (f"https://github.com/Dao-AILab/flash-attention/releases/download/v{v}/"
             f"flash_attn-{v}+cu12torch{tmm}cxx11abi{'TRUE' if abi == '1' else 'FALSE'}-{py}-{py}-linux_x86_64.whl")
        try:
            vpip("--no-deps", "-c", constraints, u)    # --no-deps: its only deps (torch, einops) are in
        except subprocess.CalledProcessError:
            print("   no such wheel")
            continue
        ok, why = _fa_loads()
        if ok:
            wheel = u
            break
        print("   installs but does not load against torch", tv, "->", *why)
        sh(PY, "-m", "pip", "uninstall", "-y", "-q", "flash-attn")
    if wheel:
        print("flash-attn wheel:", wheel.rsplit("/", 1)[1])
    elif FLASH_ATTN_FROM_SOURCE:
        vpip("-c", constraints, "ninja", "packaging", "wheel", "psutil", "setuptools")
        # FORCE_BUILD: otherwise setup.py fetches the same prebuilt wheel that just failed to load
        sh(PY, "-m", "pip", "install", "--no-deps", "--no-build-isolation", "-c", constraints,
           "flash-attn==2.7.4.post1", env=dict(VENV_ENV, MAX_JOBS="8", FLASH_ATTENTION_FORCE_BUILD="TRUE"))
        ok, why = _fa_loads()
        if not ok:
            raise RuntimeError(f"flash-attn built from source does not load: {why}")
    else:
        raise RuntimeError(f"No prebuilt flash-attn wheel loads against torch {tv} / {py} / cxx11abi={abi}. "
                           "Set FLASH_ATTN_FROM_SOURCE = True (about 1-2 h) and rerun this cell.")
print("venv:", check_pins())

check = r"""
import torch
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| cxx11abi", torch._C._GLIBCXX_USE_CXX11_ABI)
assert torch.__version__.startswith("2.6."), "torch must be 2.6.x: the eviction does not compile on 2.7.1"
assert torch.cuda.is_available(), "CUDA is not available inside the venv"
import triton, transformers, flash_attn, datasets
print("triton", triton.__version__, "| transformers", transformers.__version__,
      "| flash_attn", flash_attn.__version__, "| datasets", datasets.__version__)
from flash_attn import flash_attn_func
x = torch.randn(1, 256, 8, 128, device="cuda", dtype=torch.float16)
flash_attn_func(x, x, x, causal=True); torch.cuda.synchronize()
f = torch.compile(lambda t: (t * 2).sin().sum(-1))     # the eviction is compiled-or-raise
f(torch.randn(64, 64, device="cuda")); torch.cuda.synchronize()
from utils.cache_factory import assert_transformers_version_supported
assert_transformers_version_supported()
print("environment ok")
"""
r = subprocess.run([PY, "-c", check], cwd=REPO, env=VENV_ENV, capture_output=True, text=True)
print(r.stdout, r.stderr[-4000:] if r.returncode else "")
if r.returncode != 0:
    raise RuntimeError("environment check failed inside the venv; see the output above")
MARK.write_text(STACK_ID)

# The scoring cells (section 8) run in this kernel: they need the LongBench metric packages here too.
missing = [m for m in ("rouge", "fuzzywuzzy", "Levenshtein") if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([KERNEL_PY, "-m", "pip", "install", "-q", "rouge", "fuzzywuzzy", "python-Levenshtein"],
                   check=True)
print("kernel:", sys.version.split()[0], "| jobs:", PY)

## 3. Model and data

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download
HF_TOKEN = _secret("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Add a Colab secret HF_TOKEN with access to meta-llama/Llama-3.1-8B-Instruct.")
assert "instruct" in Path(MODEL_DIR).name.lower(), "MODEL_DIR's name must contain 'instruct'"
snapshot_download(MODEL_ID, local_dir=MODEL_DIR, token=HF_TOKEN,
                  allow_patterns=["*.json", "*.safetensors", "tokenizer*"])

LB_DIR = REPO / "data" / "longbench"
LB_DIR.mkdir(parents=True, exist_ok=True)
need = [t for t in TASKS if not (LB_DIR / f"{t}.jsonl").exists()]
if need:
    zp = hf_hub_download("THUDM/LongBench", "data.zip", repo_type="dataset", token=HF_TOKEN)
    with zipfile.ZipFile(zp) as z:                      # members are data/<task>.jsonl
        names = z.namelist()
        for t in need:
            member = next(n for n in names if n == f"{t}.jsonl" or n.endswith(f"/{t}.jsonl"))
            with z.open(member) as src, open(LB_DIR / f"{t}.jsonl", "wb") as dst:
                shutil.copyfileobj(src, dst)
for t in TASKS:
    ids = [json.loads(l)["_id"] for l in open(LB_DIR / f"{t}.jsonl")]
    assert len(ids) == len(set(ids)), f"{t}: duplicate _id, paired comparisons would be wrong"
    print(f"{t:12s} {len(ids)} examples, unique _id")

ENV = dict(VENV_ENV, PYTHONHASHSEED="0", CUBLAS_WORKSPACE_CONFIG=":4096:8",
           TOKENIZERS_PARALLELISM="false", LONGBENCH_LOCAL_DIR=str(LB_DIR),
           HF_TOKEN=HF_TOKEN, PYTHONUNBUFFERED="1")

if RUN_GSM8K and RUN == "extended" and not (REPO / "data/gsm8k_cot").exists():
    subprocess.run([PY, "-m", "data.gsm8k_loader", "--out", "data/gsm8k_cot"],
                   cwd=REPO, env=ENV, check=True)

## 4. Readiness check (refuses to start if anything is missing)

1. The gate is wired: `fused_gate` has non-test callers.
2. Every config is at the operating point.
3. The completion knob exists on `CacheConfig`.
4. The collector has `--completion-rungs`.
5. Preflight tests pass on this GPU:
   * the completion-ablation suite, i.e. the epilogue against the CPU oracle, the plumbing and the guard;
   * the gate/fill suites;
   * the sm80 compile tests.

In [ ]:
probe = r"""
import json, dataclasses, subprocess, pathlib
from utils.config import CacheConfig
fields = {f.name for f in dataclasses.fields(CacheConfig)}
callers = subprocess.run("grep -rn 'fused_gate' --include='*.py' . | grep -v '^./tests/'",
                         shell=True, capture_output=True, text=True).stdout.splitlines()
src = pathlib.Path("modules/evaluation/observation_collector.py").read_text()
knobs = ("quant_completion", "quant_completion_scale", "quant_completion_sigma",
         "quant_completion_seed", "quant_completion_scores")
print(json.dumps({"knobs": {k: k in fields for k in knobs},
                  "gate_callers": [c for c in callers if "def fused_gate" not in c],
                  "exp1b_rungs": "--completion-rungs" in src}))
"""
info = json.loads(subprocess.check_output([PY, "-c", probe], cwd=REPO, env=ENV, text=True))
op = subprocess.run([PY, "scripts/check_operating_point.py"], cwd=REPO, env=ENV,
                    capture_output=True, text=True)
READY = {"gate_wired": bool(info["gate_callers"]), "operating_point": op.returncode == 0,
         "knob": all(info["knobs"].values()), "exp1b": info["exp1b_rungs"]}
print(f"commit {SHA[:10]}: " + ", ".join(f"{k}={v}" for k, v in READY.items()))
if not all(READY.values()):
    print(op.stdout[-2000:])
    raise RuntimeError("This commit cannot run the experiment: " +
                       ", ".join(k for k, v in READY.items() if not v) +
                       ". Nothing was run: control arms from this commit could not be combined with "
                       "variant arms from a later one.")
if RUN_PREFLIGHT_TESTS:
    t = subprocess.run([PY, "-m", "pytest", "-q", "-x", "-p", "no:cacheprovider",
                        "tests/test_completion_ablation.py", "tests/test_gate_fill_calibration.py",
                        "tests/test_gate_union_is_per_head.py", "tests/test_kernel_compiles.py"],
                       cwd=REPO, env=dict(ENV, ATEN_CPU_CAPABILITY="default"),  # scalar CPU codegen for the
                       capture_output=True, text=True)                           # CPU-side cache tests only
    print(t.stdout[-1200:])
    if t.returncode != 0:
        raise RuntimeError("Preflight tests failed on this GPU. See the output above.")
print("ready")

## 5. Arms and the job runner

In [ ]:
LOGS, RUNS = ROOT / "logs", Path("/content/runs")

class Blocked(Exception):
    pass

def run(cmd, log_name, env=None, log_dir=None):
    """`env` and `log_dir` default to ENV and ROOT/logs, the original run; Part II passes its own."""
    cmd = [str(c) for c in cmd]
    log = (Path(log_dir) if log_dir else LOGS) / f"{log_name}.log"
    print("$", " ".join(cmd))
    t0, tail = time.time(), []
    keep = ("examples done", "read gate", "OOM", "Error", "error:", "NOT-GATED", "Traceback", "score=")
    with open(log, "w") as fh:
        p = subprocess.Popen(cmd, cwd=REPO, env=ENV if env is None else env, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            fh.write(line)
            tail = (tail + [line])[-60:]
            if any(k in line for k in keep):
                print("   ", line.rstrip()[:200])
        rc = p.wait()
    if rc != 0:
        print("".join(tail))
        raise RuntimeError(f"{log_name} failed (rc={rc}). Full log: {log}")
    print(f"    finished in {(time.time() - t0) / 60:.1f} min")

NATIVE = {"mode": "native", "scale": 1.0, "sigma": 0.0, "seed": 0, "scores": "output"}
def C(**kw):
    return {"mode": "calibrated", "scale": 1.0, "sigma": 0.0, "seed": 0, "scores": "output", **kw}

ARMS = {}
def arm(name, stage, exp, gate=GATE, comp=NATIVE, sigma_mult=None, floor=False):
    ARMS[name] = dict(stage=stage, exp=exp, gate=gate, comp=comp, sigma_mult=sigma_mult, floor=floor)

# core: one point per question end to end. Experiment 1 covers every level on matched states.
arm("cal",          "core", "control")
arm("off",          "core", "2", comp=C(mode="off"))
arm("uncal",        "core", "3", comp=C(mode="uncalibrated"))
arm("scale2",       "core", "4A", comp=C(scale=2.0))           # over-crediting: the direction section 11 found harmful
arm("noise1x_s0",   "core", "4B", comp=C(seed=0), sigma_mult=1)
arm("cal_repeat",   "core", "floor", floor=True)               # run-to-run noise
arm("cal_ep",       "core", "floor", comp=C(), floor=True)     # the epilogue at identity
# extended
arm("ceiling_g1.0", "extended", "2", gate=1.0)                 # nothing unread
for c in (0.5, 0.8, 1.25, 4.0):
    arm(f"scale{c:g}", "extended", "4A", comp=C(scale=c))
for s in (1, 2):
    arm(f"noise1x_s{s}", "extended", "4B", comp=C(seed=s), sigma_mult=1)
for mult in (2, 4):
    arm(f"noise{mult}x_s0", "extended", "4B", comp=C(seed=0), sigma_mult=mult)
arm("stress_g0.10_cal", "extended", "2", gate=0.10)
arm("stress_g0.10_off", "extended", "2", gate=0.10, comp=C(mode="off"))
for c in (0.5, 2.0):
    arm(f"coupled_scale{c:g}", "extended", "4C", comp=C(scale=c, scores="coupled"))

SIGMA_FILE = ROOT / "exp1" / "sigma_hat.json"

def n_for(a):
    return NUM_SAMPLES_FLOOR if a["floor"] else NUM_SAMPLES

def resolved_comp(a):
    c = dict(a["comp"])
    if a["sigma_mult"]:
        if not SIGMA_FILE.exists():
            raise Blocked("noise arms need sigma_hat_U from Experiment 1. Run section 6 first")
        c["sigma"] = a["sigma_mult"] * json.loads(SIGMA_FILE.read_text())["sigma_hat_U"]
    return c

def overrides(a, comp):
    ov = [f"model.name={MODEL_DIR}", f"cache.quant_gate_ratio={a['gate']}"]
    if comp["mode"] != "native":
        ov += [f"cache.quant_completion={comp['mode']}", f"cache.quant_completion_scale={comp['scale']}",
               f"cache.quant_completion_sigma={comp['sigma']}", f"cache.quant_completion_seed={comp['seed']}",
               f"cache.quant_completion_scores={comp['scores']}"]
    return ov

def _same(g, v):
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        return g is not None and abs(float(g) - float(v)) <= 1e-9 * max(1.0, abs(float(v)))
    return g == v

def verify(meta, a, comp, n=None):
    """Everything a sidecar must say for its number to belong to this arm."""
    errs = []
    rg = meta.get("read_gate") or {}
    if rg.get("verdict") != "gated":
        errs.append(f"read_gate verdict {rg.get('verdict')!r}, need 'gated'")
    rf = rg.get("read_fraction")
    if rf is None or not (a["gate"] - 1e-6 <= rf <= min(1.0, a["gate"] + 0.15) + 1e-6):
        errs.append(f"realised read fraction {rf} for requested ratio {a['gate']}")
    for k, v in OPERATING_POINT.items():
        if k in meta and not _same(meta[k], v):
            errs.append(f"{k} = {meta[k]!r}, expected {v!r}")
    got = meta.get("quant_completion")
    if not isinstance(got, dict):
        errs.append("no quant_completion record in the sidecar")
    else:
        for k, v in comp.items():
            if not _same(got.get(k), v):
                errs.append(f"quant_completion.{k} = {got.get(k)!r}, expected {v!r}")
    applied, gated = rg.get("completion_applied"), rg.get("gated")
    want = 0 if comp["mode"] == "native" else gated
    if applied is None or applied != want or (comp["mode"] != "native" and not gated):
        errs.append(f"completion_applied = {applied}, gated = {gated}: the epilogue must run on every "
                    "gated layer of a variant arm and on none of a native one")
    if n is not None and "num_samples_requested" in meta and meta["num_samples_requested"] != n:
        errs.append(f"num_samples_requested = {meta['num_samples_requested']!r}, expected {n!r}")
    return errs

def _refuse(files, tag, errs, base=None):
    dst = (Path(base) if base else ROOT / "refused") / tag
    dst.mkdir(parents=True, exist_ok=True)
    for f in files:
        if Path(f).exists():
            shutil.move(str(f), dst / Path(f).name)
    raise RuntimeError(f"REFUSED {tag}:\n  " + "\n  ".join(errs) + f"\n  files moved to {dst}")

def keep_or_write(path, write):
    """`write(path)`, unless ORIGINAL_FROZEN and the file already exists: the original analysis is never
    rewritten."""
    path = Path(path)
    if ORIGINAL_FROZEN and path.exists():
        print(f"   kept {path.relative_to(ROOT)} as written (ORIGINAL_FROZEN)")
        return False
    write(path)
    return True

def lb_arm(name):
    """Every pending task of one arm in ONE process (one model load), then each task's
    sidecar verified and copied to Drive. A task that finished before a crash is kept."""
    a = ARMS[name]
    comp, n = resolved_comp(a), n_for(a)
    dst = ROOT / "longbench" / name
    dst.mkdir(parents=True, exist_ok=True)
    todo = []
    for t in TASKS:
        meta_f, pred_f = dst / f"{t}.meta.json", dst / f"{t}.jsonl"
        if meta_f.exists() and pred_f.exists():     # the sidecar is written last: it is the done-marker
            errs = verify(json.loads(meta_f.read_text()), a, comp, n)
            if errs:
                raise RuntimeError(f"STALE {name}/{t} on Drive does not match this arm:\n  "
                                   + "\n  ".join(errs) + "\n  Move it away to rerun.")
        else:
            todo.append(t)
    if not todo:
        return "done"
    local = RUNS / "longbench" / name
    shutil.rmtree(local, ignore_errors=True)    # the runner's own resume trusts a partial jsonl
    local.mkdir(parents=True)
    cmd = [PY, "main.py", "--config", "configs/longbench_ours_flash_attn.yaml", "--override",
           f"longbench.datasets=[{','.join(todo)}]", f"longbench.output_dir={local}",
           f"longbench.num_samples={n}", "longbench.resume=false",
           f"longbench.skip_oom={str(SKIP_OOM).lower()}", *overrides(a, comp)]
    failure = None
    try:
        run(cmd, f"lb_{name}")
    except RuntimeError as e:
        failure = e
    for t in todo:
        mf, pf = local / f"{t}.meta.json", local / f"{t}.jsonl"
        if not (mf.exists() and pf.exists()):
            continue
        errs = verify(json.loads(mf.read_text()), a, comp, n)
        if errs:
            _refuse([pf, mf], f"{name}_{t}", errs)
        shutil.copy(pf, dst / f"{t}.jsonl")
        shutil.copy(mf, dst / f"{t}.meta.json")
    if failure is not None:
        raise failure
    return "ran"

GSM8K_ARMS = ["cal", "off", "uncal", "scale0.5", "scale2"]

def gsm8k_job(name):
    a = ARMS[name]
    comp = resolved_comp(a)
    dst = ROOT / "gsm8k" / name
    if (dst / "meta.json").exists() and (dst / "predictions.jsonl").exists():
        errs = verify(json.loads((dst / "meta.json").read_text()), a, comp, GSM8K_NUM_SAMPLES)
        if errs:
            raise RuntimeError(f"STALE gsm8k/{name}:\n  " + "\n  ".join(errs))
        return "done"
    local = RUNS / "gsm8k" / name
    shutil.rmtree(local, ignore_errors=True)
    run([PY, "main.py", "--config", "configs/gsm8k_budget20.yaml", "--override",
         f"gsm8k.output_dir={local}", f"gsm8k.num_samples={GSM8K_NUM_SAMPLES}",
         f"gsm8k.skip_oom={str(SKIP_OOM).lower()}", *overrides(a, comp)], f"gsm8k_{name}")
    meta = json.loads((local / "meta.json").read_text())
    errs = verify(meta, a, comp, GSM8K_NUM_SAMPLES)
    if errs:
        _refuse([local / "predictions.jsonl", local / "meta.json"], f"gsm8k_{name}", errs)
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copy(local / "predictions.jsonl", dst / "predictions.jsonl")
    shutil.copy(local / "meta.json", dst / "meta.json")
    return "ran"

def status_table(stage_max):
    names = [n for n, a in ARMS.items() if STAGES[a["stage"]] <= stage_max]
    loads = 0
    rows, remaining_s = [], 0.0
    eps = {}
    for t in TASKS:   # measured examples/second per task, from any finished native arm
        for n in ("cal", "ceiling_g1.0", "off"):
            f = ROOT / "longbench" / n / f"{t}.meta.json"
            if f.exists():
                eps[t] = json.loads(f.read_text()).get("examples_per_second")
                break
    print(f"\n{'arm':20s}{'set':>9s}" + "".join(f"{t:>13s}" for t in TASKS))
    for n in names:
        cells = []
        for t in TASKS:
            done = (ROOT / "longbench" / n / f"{t}.meta.json").exists()
            cells.append("done" if done else "-")
            if not done and eps.get(t):
                remaining_s += n_for(ARMS[n]) / eps[t] * (1.0 if ARMS[n]["comp"]["mode"] == "native" else 1.2)
        print(f"{n:20s}{ARMS[n]['stage']:>9s}" + "".join(f"{c:>13s}" for c in cells))
        loads += "-" in cells
    if eps:
        remaining_s += 90 * loads                  # one model load + compile warm-up per pending arm
        print(f"\nprojected remaining for RUN = {RUN!r}: {remaining_s / 3600:.1f} h "
              "(measured ex/s of finished native jobs; +20% for epilogue arms; ~90 s load per arm)")

def run_grid(stage_max):
    for n, a in ARMS.items():
        if STAGES[a["stage"]] > stage_max:
            continue
        try:
            if lb_arm(n) == "ran":
                status_table(stage_max)            # progress, and the re-projected remaining time
        except Blocked as e:
            print(f"BLOCKED {n}: {e}")
    status_table(stage_max)

## 6. Experiment 1: every variant on the same decode states

One collector run per task on the shipped method, with `--completion-rungs`. At every gated (step, layer, query head)
the collector holds the dequantized int2 tier and the full-KV shadow, so it computes **on the same state**:
* the exact int2 mass of every window over `Z_A`, `m*(u)`;
* the calibrated and uncalibrated masses;
* the output of each variant: `off`, `uncal`, `cal`, five scales, four noise levels, the **mass oracle** (`m*` with the
  stored centroid), the **value oracle** and the **full oracle** (= attention over every held window).

Output errors are against `o_held`, which removes eviction and int2 error and isolates the completion term.

| Statistic | Meaning |
|---|---|
| `unread` | share of held attention on unread int2 windows: what the completion stands in for |
| `E_w`, `E_unw` | `|log m̃_cal − log m*|` over unread windows, mass-weighted and unweighted (the reviewer's `E_logm`; the unweighted form is dominated by near-empty windows) |
| `bias_w` | mean signed log error on U. `δ` is fitted on R, so the R mean is 0 by construction; this is the selection bias |
| `tot_err` | `log Σ_U m̃_cal − log Σ_U m*`, the error on the total unread mass. `c_hat = exp(median)` is marked on 4A |
| `sd_U` | residual spread on U. Its median is `σ̂_U`, the unit of the noise arms |
| `E_w_uncal`, `uncal_factor` | the same for `uncal`; `exp(uncal_shift)` is the per-step common factor `uncal` applies to `cal` |
| rungs | rel. L2 output error vs `o_held`, normalised by `|o_full|` like the collector's ladder |

Sanity gates that refuse the data:
* `full_oracle` must reproduce the collector's own `o_held` (≤ 1e-3);
* `cal` must reproduce the kernel's real output: median `cal_vs_actual` ≤ `EXP1_CAL_VS_KERNEL_MAX`. fp16 storage and the kernel's base-2 softmax account for roughly 1e-3.

Statistics are taken per query head first, then aggregated over heads. Windows below 1e-6 mass are left out of the unweighted
statistics. **Caveat:** prompts are the first `EXP1_PREFILL` tokens of each task's documents plus a greedy continuation, not
the templated task prompt. Collector zips (~0.5–1 GB per prompt) stay on local disk; only the analysed results go to Drive.

**With `ORIGINAL_FROZEN = True`** this section only loads the results on Drive. A task with no results is reported, not run, and
`sigma_hat.json` is never rewritten: it is written once, when all four tasks are in. Part II calls the same job at ρ = 0.10 with
its own folders.

In [ ]:
import numpy as np, pandas as pd

FLOOR = 1e-6
RUNGS = None   # read from the zip's meta

def iter_zip(path, keys):
    with zipfile.ZipFile(path) as zf:
        meta = json.loads(zf.read("meta.json"))
        for s in meta["samples"]:
            with np.load(io.BytesIO(zf.read(s["file"])), allow_pickle=False) as z:
                yield {k: z[k] for k in keys}

def _nanmean(x, w=None, axis=-1):
    with np.errstate(invalid="ignore", divide="ignore"):
        if w is None:
            return np.nanmean(x, axis=axis) if np.isfinite(x).any() else np.nan
        return np.nansum(x * w, axis=axis) / np.nansum(w, axis=axis)

def _head_step(tier_tl, go_tl, lms, err, shift, cerr, cva, rep):
    """One (step, layer): a row of statistics per query head (NaN where undefined)."""
    q = (tier_tl == 1)[None, :]
    op = np.repeat(go_tl, rep, axis=0)
    R, U = q & op, q & ~op
    lms = lms.astype(np.float64); e = err.astype(np.float64)
    fin = np.isfinite(lms) & np.isfinite(e)
    m = np.where(fin, np.exp(np.where(fin, lms, -np.inf)), 0.0)
    Uw, Uf = U & fin, U & fin & (m >= FLOOR)
    nU = Uf.sum(1)
    with np.errstate(invalid="ignore", divide="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        wU = np.where(Uw, m, 0.0); sw = wU.sum(1)
        e_unc = e + shift[:, None]
        out = {
            "unread": sw / (1.0 + sw),
            "E_w": (wU * np.abs(np.where(Uw, e, 0))).sum(1) / sw,
            "E_unw": np.where(nU > 0, (np.abs(np.where(Uf, e, 0))).sum(1) / nU, np.nan),
            "bias_w": (wU * np.where(Uw, e, 0)).sum(1) / sw,
            "bias_unw": np.where(nU > 0, np.where(Uf, e, 0).sum(1) / nU, np.nan),
            "tot_err": np.log((wU * np.exp(np.where(Uw, e, 0))).sum(1)) - np.log(sw),
            "sd_U": np.where(nU >= 4, np.nanstd(np.where(Uf, e, np.nan), axis=1, ddof=1), np.nan),
            "E_w_uncal": (wU * np.abs(np.where(Uw, e_unc, 0))).sum(1) / sw,
            "uncal_factor": np.exp(shift),
            "cal_vs_actual": cva.astype(np.float64),
        }
    valid = (R.sum(1) > 0) & (Uw.sum(1) > 0)
    for k in out:
        out[k] = np.where(valid & np.isfinite(out[k]), out[k], np.nan)
    rung = np.where(valid[:, None], cerr[..., 0].astype(np.float64), np.nan)     # vs o_held
    rung_full = np.where(valid[:, None], cerr[..., 1].astype(np.float64), np.nan)
    return out, rung, rung_full

EXP1_KEYS = ("tier", "gate_open", "gate_fired", "comp_log_mstar", "comp_logerr_cal",
             "comp_uncal_shift", "comp_err", "comp_cal_vs_actual")

def exp1_analyse(zp):
    with zipfile.ZipFile(zp) as zf:
        meta = json.loads(zf.read("meta.json"))
    rungs = meta["completion_rungs"]
    stats, rg, rgf, ids = {}, [], [], []
    for si, s in enumerate(iter_zip(zp, EXP1_KEYS)):
        T, L, H, _ = s["comp_log_mstar"].shape
        rep = H // s["gate_open"].shape[2]
        for t in range(1, T):
            for l in np.flatnonzero(s["gate_fired"][t]):
                o, r, rf = _head_step(s["tier"][t, l], s["gate_open"][t, l], s["comp_log_mstar"][t, l],
                                      s["comp_logerr_cal"][t, l], s["comp_uncal_shift"][t, l],
                                      s["comp_err"][t, l], s["comp_cal_vs_actual"][t, l], rep)
                for k, v in o.items():
                    stats.setdefault(k, []).append(v.astype(np.float32))
                rg.append(r.astype(np.float32)); rgf.append(rf.astype(np.float32))
                ids.append(np.stack([np.full(H, l), np.arange(H)], 1))
    ids = np.concatenate(ids)
    df = pd.DataFrame({k: np.concatenate(v) for k, v in stats.items()})
    df["layer"], df["head"] = ids[:, 0], ids[:, 1]
    R = pd.DataFrame(np.concatenate(rg), columns=rungs)
    Rf = pd.DataFrame(np.concatenate(rgf), columns=rungs)
    R["layer"] = Rf["layer"] = ids[:, 0]
    R["head"] = Rf["head"] = ids[:, 1]
    return meta, df, R, Rf

def per_head(df):
    return df.groupby(["layer", "head"]).median()

def summarise(df):
    ph = per_head(df)
    worst = {k: (ph[k].abs().max() if k in ("bias_w", "bias_unw", "tot_err") else ph[k].max()) for k in ph}
    return pd.DataFrame({"median head": ph.median(), "p90 head": ph.quantile(0.9), "worst head": pd.Series(worst)})

def exp1_job(task, gate=GATE, out_dir=None, tag="exp1", log_dir=None, refused=None):
    """The keyword arguments default to the original Experiment 1 (rho = GATE, ROOT/exp1). Part II passes
    rho = 0.10 and its own folders, so the two never share a file."""
    out_dir = Path(out_dir) if out_dir else ROOT / "exp1"
    res = out_dir / f"{task}_results.json"
    if res.exists():
        return json.loads(res.read_text())
    if ORIGINAL_FROZEN and out_dir == ROOT / "exp1":
        print(f"exp1 {task}: no results on Drive, and ORIGINAL_FROZEN = True, so it is not run")
        return None
    out_dir.mkdir(parents=True, exist_ok=True)
    zp = RUNS / tag / f"{task}_p{EXP1_PREFILL}_n{EXP1_GEN}_k{EXP1_PROMPTS}.zip"
    zp.parent.mkdir(parents=True, exist_ok=True)
    if not zp.exists():
        run(["bash", "scripts/collect_observations.sh", "--model-path", MODEL_DIR, "--dataset", task,
             "--window-size", OPERATING_POINT["window_size"], "--cache-budget", OPERATING_POINT["cache_budget"],
             "--gate-ratio", gate, "--quant-ratio", OPERATING_POINT["quant_ratio"],
             "--quant-budget-mode", OPERATING_POINT["quant_budget_mode"],
             "--prefill", EXP1_PREFILL, "--gen", EXP1_GEN, "--num-samples", EXP1_PROMPTS,
             "--completion-rungs", "--out", zp], f"{tag}_{task}", log_dir=log_dir)
    meta, df, R, Rf = exp1_analyse(zp)
    errs = []
    if (meta.get("read_gate") or {}).get("verdict") != "gated":
        errs.append(f"collector read_gate verdict {(meta.get('read_gate') or {}).get('verdict')!r}")
    rf = (meta.get("read_gate") or {}).get("read_fraction")
    if rf is None or not (gate - 1e-6 <= rf <= min(1.0, gate + 0.15) + 1e-6):
        errs.append(f"collector realised read fraction {rf} for requested ratio {gate}")
    fo = float(np.nanmax(R["full_oracle"])) if len(R) else float("nan")
    if not fo <= 1e-3:
        errs.append(f"full_oracle differs from the collector's o_held by up to {fo:.2e}: the rung arithmetic is wrong")
    cva = float(np.nanmedian(df["cal_vs_actual"]))
    if not cva <= EXP1_CAL_VS_KERNEL_MAX:
        errs.append(f"the recomputed calibrated output differs from the kernel's real output by {cva:.2e} "
                    f"(median) > {EXP1_CAL_VS_KERNEL_MAX}: the rungs would describe a different computation "
                    "from the one that ran")
    if errs:
        _refuse([zp], f"{tag}_{task}", errs, base=refused)
    s = summarise(df)
    rung_tab = pd.DataFrame({"median head": R.groupby(["layer", "head"]).median().median(),
                             "p90 head": R.groupby(["layer", "head"]).median().quantile(0.9),
                             "vs full-KV, median head": Rf.groupby(["layer", "head"]).median().median()}
                            ).drop(index=["layer", "head"], errors="ignore")
    out = {"task": task, "head_steps": int(len(df)), "summary": s.to_dict(), "rungs": rung_tab.to_dict(),
           "sigma_hat_U": float(np.nanmedian(df["sd_U"])), "tot_err_median": float(np.nanmedian(df["tot_err"])),
           "uncal_factor_median": float(np.nanmedian(df["uncal_factor"])),
           "cal_vs_actual_median": float(np.nanmedian(df["cal_vs_actual"])),
           "cal_vs_actual_p99": float(np.nanpercentile(df["cal_vs_actual"].dropna(), 99)) if df["cal_vs_actual"].notna().any() else None,
           "full_oracle_max": fo, "sha": SHA, "gate": gate, "read_fraction": rf,
           "collector_seconds": float(sum(s.get("seconds", 0) for s in meta.get("samples", [])))}
    per_head(df).to_csv(out_dir / f"{task}_per_head.csv")
    R.groupby(["layer", "head"]).median().to_csv(out_dir / f"{task}_rungs_per_head.csv")
    res.write_text(json.dumps(out, indent=1, default=float))
    return out

def exp1_all():
    E1 = {t: r for t in TASKS if (r := exp1_job(t)) is not None}
    if not E1:
        print("Experiment 1: no results on Drive" + (" (ORIGINAL_FROZEN = True: not run)" if ORIGINAL_FROZEN else ""))
        return E1
    for t, r in E1.items():
        print(f"\n=== {t}: {r['head_steps']} head-steps   cal vs kernel output: median {r['cal_vs_actual_median']:.2e}, "
              f"p99 {r['cal_vs_actual_p99']:.2e}   full oracle vs o_held: max {r['full_oracle_max']:.1e}")
        print(pd.DataFrame(r["summary"]).round(4).to_string())
        print(pd.DataFrame(r["rungs"]).round(5).to_string())
    sig = float(np.median([r["sigma_hat_U"] for r in E1.values()]))
    chat = float(np.exp(np.median([r["tot_err_median"] for r in E1.values()])))
    ufac = float(np.median([r["uncal_factor_median"] for r in E1.values()]))
    complete = len(E1) == len(TASKS)
    if SIGMA_FILE.exists():   # never rewritten: every noise arm on Drive ran at exactly the value it holds
        old = json.loads(SIGMA_FILE.read_text())
        if complete and abs(old["sigma_hat_U"] - sig) > 1e-9:
            raise RuntimeError(f"sigma_hat_U changed ({old['sigma_hat_U']} -> {sig}); noise arms already on Drive "
                               "were run at the old value. Keep the old Exp 1 results or start a new DRIVE_ROOT.")
    elif complete:
        SIGMA_FILE.write_text(json.dumps({"sigma_hat_U": sig, "c_hat": chat, "uncal_factor": ufac, "floor": FLOOR,
                                          "sha": SHA, "per_task": {t: {k: r[k] for k in ("sigma_hat_U",
                                          "tot_err_median", "uncal_factor_median")} for t, r in E1.items()}}, indent=2))
    print(f"\nsigma_hat_U = {sig:.3f} nats (noise-arm unit)   c_hat = {chat:.3f} (measured common-scale error of "
          f"the fill; marked on 4A)   uncal factor = {ufac:.3f} (uncal = cal x this, per step; compare with 4A)")
    for t, r in E1.items():
        m = r["rungs"]["median head"]
        print(f"{t:12s} matched states: completion removes {1 - m['cal'] / m['off']:.1%} of the read-gate output "
              f"error (off {m['off']:.4f} -> cal {m['cal']:.4f}); a perfect mass estimate would reach "
              f"{m['mass_oracle']:.4f}, the stored centroid's floor")
    return E1

E1 = exp1_all()

## 7. Experiments 2–4: the LongBench arms for the current `RUN`

With `ORIGINAL_FROZEN = True` this only prints the status of what is on Drive. Part II runs the follow-up arms.

In [ ]:
if ORIGINAL_FROZEN:
    print("ORIGINAL_FROZEN = True: the original grid is not run. What is on Drive:")
    status_table(STAGES[RUN])
else:
    run_grid(STAGES[RUN])

In [ ]:
if RUN_GSM8K and RUN == "extended" and not ORIGINAL_FROZEN:
    for n in GSM8K_ARMS:
        print(n, gsm8k_job(n))

## 8. Scoring, power check, analysis

* Per-example scores use the repo's own LongBench metric functions and first-line rule.
* Task scores follow THUDM: a null (OOM) prediction scores 0.
* Paired deltas use examples that are non-null in both arms, matched by `_id`. Each comes with a 95% bootstrap CI.
* **Behavioural change** is reported against `cal`:
  * the share of examples whose prediction text changed at all;
  * the mean text similarity.

  Read it against the two floors, `cal_repeat` and `cal_ep`.
* **Power check:** the minimum detectable effect (80% power, two-sided 5%) at the current N, from the measured paired spread.
  End to end, the core run only resolves large changes. Experiment 1 carries the precise answer.
* Every sidecar is re-verified, and a file that fails is not scored.
* With `ORIGINAL_FROZEN = True`, sections 8 and 9 print their tables, and the files already in `results/` are kept
  as written. Part II's analysis (sections 12–15) writes to `followup/results/`.

In [ ]:
os.chdir(REPO)
sys.path.insert(0, str(REPO))
from modules.evaluation.longbench_scoring import METRIC_FN_REGISTRY, _FIRST_LINE_DATASETS
D2M = json.loads((REPO / "data/longbench_configs/dataset2metric.json").read_text())

def example_scores(jsonl, task):
    fn, out, preds = METRIC_FN_REGISTRY[D2M[task]], {}, {}
    for line in open(jsonl, encoding="utf-8"):
        if not line.strip():
            continue
        ex = json.loads(line)
        pred = ex.get("pred")
        preds[ex["_id"]] = pred
        if pred is None:
            out[ex["_id"]] = None
            continue
        if task in _FIRST_LINE_DATASETS:
            pred = pred.lstrip("\n").split("\n")[0]
        out[ex["_id"]] = 100.0 * max((fn(pred, gt, all_classes=ex.get("all_classes") or [])
                                      for gt in ex.get("answers", [])), default=0.0)
    return out, preds

SCORES, PREDS, SKIPPED = {}, {}, []
for name, a in ARMS.items():
    for t in TASKS:
        d = ROOT / "longbench" / name
        if not (d / f"{t}.meta.json").exists():
            continue
        try:
            comp = resolved_comp(a)
        except Blocked:
            continue
        errs = verify(json.loads((d / f"{t}.meta.json").read_text()), a, comp, n_for(a))
        if errs:
            SKIPPED.append((name, t, errs))
            continue
        SCORES[(name, t)], PREDS[(name, t)] = example_scores(d / f"{t}.jsonl", t)
for s in SKIPPED:
    print("NOT SCORED", s)

rng = np.random.default_rng(0)
def paired(x, y):
    ids = sorted(i for i in set(x) & set(y) if x[i] is not None and y[i] is not None)
    return np.array([x[i] for i in ids]), np.array([y[i] for i in ids]), ids

def boot_dist(d):
    idx = rng.integers(0, len(d), (BOOTSTRAP_ITERS, len(d)))
    return d[idx].mean(1)

arms_present = [n for n in ARMS if any((n, t) in SCORES for t in TASKS)]
tab = pd.DataFrame({t: {n: (np.mean([v if v is not None else 0.0 for v in SCORES[(n, t)].values()])
                            if (n, t) in SCORES else np.nan) for n in arms_present} for t in TASKS})
tab["average"] = tab[TASKS].mean(axis=1)
# The floors run only the first NUM_SAMPLES_FLOOR examples, so their task scores are on a different (smaller)
# example set than cal's row. This row is cal on exactly the floors' examples: the one to read them against.
REF_FLOOR = f"cal_first{NUM_SAMPLES_FLOOR}"
fl = next((n for n in ("cal_repeat", "cal_ep") if all((n, t) in SCORES for t in TASKS)), None)
if fl and all(("cal", t) in SCORES for t in TASKS):
    vals = [np.mean([SCORES[("cal", t)].get(i) or 0.0 for i in SCORES[(fl, t)]]) for t in TASKS]
    tab.loc[REF_FLOOR] = [*vals, float(np.mean(vals))]
print(f"Task scores (THUDM convention; {NUM_SAMPLES} examples per task; the floors and {REF_FLOOR} "
      f"use the first {NUM_SAMPLES_FLOOR}):")
print(tab.round(2).to_string())

rows = []
for n in arms_present:
    if n == "cal" or not all((n, t) in SCORES and ("cal", t) in SCORES for t in TASKS):
        continue
    dists, means, sds, ns, row = [], [], [], [], {"arm": n, "exp": ARMS[n]["exp"]}
    same, sim = [], []
    for t in TASKS:
        x, y, ids = paired(SCORES[(n, t)], SCORES[("cal", t)])
        d = x - y
        dists.append(boot_dist(d)); means.append(d.mean()); sds.append(d.std(ddof=1)); ns.append(len(d))
        lo, hi = np.percentile(dists[-1], [2.5, 97.5])
        row[t] = f"{d.mean():+.2f} [{lo:+.2f},{hi:+.2f}]"
        pa, pb = PREDS[(n, t)], PREDS[("cal", t)]
        same += [pa[i] == pb[i] for i in ids]
        sim += [difflib.SequenceMatcher(None, pa[i], pb[i]).ratio() for i in ids]
    row["avg_delta"] = float(np.mean(means))
    row["avg_lo"], row["avg_hi"] = np.percentile(np.mean(dists, 0), [2.5, 97.5])
    row["pred_changed"] = 1.0 - float(np.mean(same))
    row["pred_similarity"] = float(np.mean(sim))
    se_avg = math.sqrt(sum((s / math.sqrt(k)) ** 2 for s, k in zip(sds, ns))) / len(TASKS)
    row["MDE80_avg"] = 2.80 * se_avg
    row["n_paired"] = int(np.sum(ns))
    rows.append(row)
DELTA = pd.DataFrame(rows).set_index("arm") if rows else pd.DataFrame()
if len(DELTA):
    print("\nPaired Δ vs cal, points [95% CI]. pred_changed / pred_similarity: behavioural change vs cal "
          "(read against cal_repeat and cal_ep). MDE80_avg: the smallest true change of the 4-task average this N "
          "detects with 80% power.")
    print(DELTA[[*TASKS, "avg_delta", "avg_lo", "avg_hi", "pred_changed", "pred_similarity",
                 "MDE80_avg", "n_paired"]].round(3).to_string())
    keep_or_write(ROOT / "results" / "paired_deltas.csv", DELTA.to_csv)
    if "off" in DELTA.index:
        mde = DELTA.loc["off", "MDE80_avg"]
        print(f"\nPOWER CHECK: at the current N, a change of the 4-task average smaller than {mde:.2f} points "
              "is not reliably detectable end to end (it scales with 1/sqrt(NUM_SAMPLES)). The fine-grained "
              "sensitivity is Experiment 1's matched-state curve.")
keep_or_write(ROOT / "results" / "task_scores.csv", tab.to_csv)

if all(("off", t) in SCORES and ("ceiling_g1.0", t) in SCORES for t in TASKS):
    num, den = [], []
    for t in TASKS:
        ids = sorted(set.intersection(*[{i for i, v in SCORES[(n, t)].items() if v is not None}
                                        for n in ("cal", "off", "ceiling_g1.0")]))
        c = np.array([SCORES[("cal", t)][i] for i in ids])
        o = np.array([SCORES[("off", t)][i] for i in ids])
        g = np.array([SCORES[("ceiling_g1.0", t)][i] for i in ids])
        idx = rng.integers(0, len(ids), (BOOTSTRAP_ITERS, len(ids)))
        num.append((c - o)[idx].mean(1)); den.append((g - o)[idx].mean(1))
    N, D = np.mean(num, 0), np.mean(den, 0)
    if np.percentile(D, 2.5) > 0:
        gc = N / D
        print(f"\nExperiment 2, share of the read-gate gap closed by completion: {np.median(gc):.2f} "
              f"[{np.percentile(gc, 2.5):.2f}, {np.percentile(gc, 97.5):.2f}]")
    else:
        print("\nExperiment 2 gap closed: undefined, because the ceiling does not beat 'off' significantly.")

if RUN_GSM8K and (ROOT / "gsm8k" / "cal" / "meta.json").exists():
    from modules.evaluation.gsm8k_scoring import gsm8k_score
    G = {}
    for n in GSM8K_ARMS:
        d = ROOT / "gsm8k" / n
        if (d / "meta.json").exists() and not verify(json.loads((d / "meta.json").read_text()), ARMS[n],
                                                     resolved_comp(ARMS[n]), GSM8K_NUM_SAMPLES):
            G[n] = {json.loads(l)["id"]: (None if json.loads(l).get("pred") is None else
                    100.0 * gsm8k_score(json.loads(l)["pred"], json.loads(l)["answer"]))
                    for l in open(d / "predictions.jsonl") if l.strip()}
    for n, s in G.items():
        if n != "cal":
            x, y, _ = paired(s, G["cal"])
            lo, hi = np.percentile(boot_dist(x - y), [2.5, 97.5])
            print(f"GSM8K {n:12s} Δ vs cal {np.mean(x - y):+.2f} [{lo:+.2f}, {hi:+.2f}]  (n={len(x)})")

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#52514e",
                     "axes.labelcolor": "#0b0b0b", "xtick.color": "#52514e", "ytick.color": "#52514e",
                     "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.8,
                     "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb"})
BLUE, MUTED = "#2a78d6", "#52514e"
SH = json.loads(SIGMA_FILE.read_text()) if SIGMA_FILE.exists() else {}

def _ref(ax, x, label, side="left", y=1):
    ax.axvline(x, color=MUTED, ls="--", lw=1, zorder=1)
    ax.annotate(label, (x, y), xycoords=("data", "axes fraction"), textcoords="offset points",
                xytext=(-4 if side == "left" else 4, -4 if y else 4), ha="right" if side == "left" else "left",
                va="top" if y else "bottom", color=MUTED, fontsize=9)

def _err(ax, xs, ys, lo, hi, **kw):
    o = np.argsort(xs)
    ax.errorbar(np.array(xs)[o], np.array(ys)[o], yerr=[np.array(lo)[o], np.array(hi)[o]],
                color=BLUE, lw=2, marker="o", ms=8, capsize=3, zorder=3, **kw)

floor = max([abs(DELTA.loc[n, "avg_delta"]) for n in ("cal_repeat", "cal_ep") if n in DELTA.index] or [0.0])
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
# top: end to end (Experiment 4)
ax = axes[0, 0]; ax.axhline(0, color=MUTED, lw=1, zorder=1); ax.axhspan(-floor, floor, color="#e6e5e1", lw=0)
xs, ys, lo, hi = [0.0], [0.0], [0.0], [0.0]
for c in (0.5, 0.8, 1.25, 2.0, 4.0):
    n = f"scale{c:g}"
    if n in DELTA.index:
        xs.append(np.log2(c)); ys.append(DELTA.loc[n, "avg_delta"])
        lo.append(DELTA.loc[n, "avg_delta"] - DELTA.loc[n, "avg_lo"]); hi.append(DELTA.loc[n, "avg_hi"] - DELTA.loc[n, "avg_delta"])
_err(ax, xs, ys, lo, hi)
if "off" in DELTA.index:
    x0 = min(xs) - 1.2
    ax.errorbar([x0], [DELTA.loc["off", "avg_delta"]],
                yerr=[[DELTA.loc["off", "avg_delta"] - DELTA.loc["off", "avg_lo"]],
                      [DELTA.loc["off", "avg_hi"] - DELTA.loc["off", "avg_delta"]]],
                color=BLUE, marker="s", ms=8, capsize=3, lw=0, elinewidth=2, zorder=3)
    ax.annotate("off (c = 0)", (x0, DELTA.loc["off", "avg_delta"]), textcoords="offset points",
                xytext=(8, 0), va="center", color=MUTED)
if SH:
    _ref(ax, np.log2(SH["c_hat"]), f"measured fill error\nc = {SH['c_hat']:.2f}", y=0)   # cal sits at the top
ax.set_xlabel("log2 c  (common scale on unread-window mass)"); ax.set_ylabel("Δ LongBench average vs cal (points)")
ax.set_title("4A, end to end: common-scale mass error", loc="left")

ax = axes[0, 1]; ax.axhline(0, color=MUTED, lw=1, zorder=1); ax.axhspan(-floor, floor, color="#e6e5e1", lw=0)
xs, ys, lo, hi = [0], [0.0], [0.0], [0.0]
for mult in (1, 2, 4):
    v = [DELTA.loc[f"noise{mult}x_s{s}", "avg_delta"] for s in range(3) if f"noise{mult}x_s{s}" in DELTA.index]
    if v:
        xs.append(mult); ys.append(np.mean(v)); lo.append(np.mean(v) - min(v)); hi.append(max(v) - np.mean(v))
_err(ax, xs, ys, lo, hi)
_ref(ax, 1, "measured residual\nspread " + r"$\hat{\sigma}_U$", side="right")   # cal is the top-left point
ax.set_xlabel(r"$\sigma\,/\,\hat{\sigma}_U$" + "  (bars: min–max over 3 seeds)")
ax.set_title("4B, end to end: redistribution, total preserved", loc="left")

# bottom: the same perturbations on matched states (Experiment 1)
if E1:
    rung = lambda r: float(np.median([E1[t]["rungs"]["median head"][r] for t in TASKS]))
    ax = axes[1, 0]
    xs = [np.log2(c) for c in (0.5, 0.8, 1.0, 1.25, 2.0, 4.0)]
    ys = [rung(r) for r in ("scale0.5", "scale0.8", "cal", "scale1.25", "scale2", "scale4")]
    ax.plot(xs, ys, color=BLUE, lw=2, marker="o", ms=8, zorder=3)
    for r, lab, x, ha, dy in (("off", "off", 0, "left", 3), ("uncal", "uncal", 1, "right", 3),
                              ("mass_oracle", "mass oracle", 0, "left", -3)):   # below its line: uncal is close
        ax.axhline(rung(r), color=MUTED, lw=1, ls=":", zorder=1)
        ax.annotate(lab, (x, rung(r)), xycoords=("axes fraction", "data"), textcoords="offset points",
                    xytext=(4 if ha == "left" else -4, dy), ha=ha, va="bottom" if dy > 0 else "top",
                    color=MUTED, fontsize=9)
    if SH:
        _ref(ax, np.log2(SH["c_hat"]), f"c = {SH['c_hat']:.2f}")
    ax.set_xlabel("log2 c"); ax.set_ylabel("output error vs o_held (median head)")
    ax.set_title("Exp 1, matched states: common scale", loc="left")
    ax.margins(y=0.12)                    # room for the label under the lowest reference line
    ax = axes[1, 1]
    sig = [0.25, 0.5, 1.0, 2.0]
    ax.plot([0] + sig, [rung("cal")] + [rung(f"noise{s:g}") for s in sig], color=BLUE, lw=2, marker="o", ms=8, zorder=3)
    if SH:
        _ref(ax, SH["sigma_hat_U"], r"$\hat{\sigma}_U$" + f" = {SH['sigma_hat_U']:.2f}", side="right")
    ax.set_xlabel(r"$\sigma$ (nats)"); ax.set_title("Exp 1, matched states: redistribution", loc="left")
fig.text(0.01, -0.02, "Top: grey band = largest |Δ| of the two floors (cal_repeat, cal_ep). Bottom: relative L2 error "
         "of the attention output vs reading every held window, same decode states for every point.",
         color=MUTED, fontsize=9)
fig.tight_layout()
keep_or_write(ROOT / "results" / "sensitivity.png", lambda p: fig.savefig(p, dpi=200, bbox_inches="tight"))
plt.show()

## 9. Rebuttal table and run status

In [ ]:
def cell(n, t):
    return tab.loc[n, t] if n in tab.index and not np.isnan(tab.loc[n, t]) else None

RUNG_OF = {"cal": "cal", "off": "off", "uncal": "uncal", "scale0.5": "scale0.5", "scale0.8": "scale0.8",
           "scale1.25": "scale1.25", "scale2": "scale2", "scale4": "scale4"}
label = {"cal": "Calibrated completion (QEvict)", "off": "Without completion", "uncal": "Uncalibrated completion",
         "scale0.5": "Perturbed masses, 0.5x", "scale0.8": "Perturbed masses, 0.8x",
         "scale1.25": "Perturbed masses, 1.25x", "scale2": "Perturbed masses, 2x", "scale4": "Perturbed masses, 4x",
         "noise1x_s0": "Redistribution noise, 1x sigma_hat (seed 0)", "ceiling_g1.0": "All windows read (rho = 1.0)",
         REF_FLOOR: f"Calibrated, on the floors' first {NUM_SAMPLES_FLOOR}/task (reference for the two rows below)",
         "cal_repeat": f"Calibrated, repeat (floor, first {NUM_SAMPLES_FLOOR}/task)",
         "cal_ep": f"Calibrated via epilogue (floor, first {NUM_SAMPLES_FLOOR}/task)"}
def rung_err(n):
    r = RUNG_OF.get(n)
    if not (E1 and r):
        return "—"
    return f"{np.median([E1[t]['rungs']['median head'][r] for t in TASKS]):.4f}"
lines = ["| Configuration | " + " | ".join(TASKS) + " | Average | Δ vs cal [95% CI] | output error, matched states |",
         "|---|" + "---|" * (len(TASKS) + 3)]
for n, lab in label.items():
    vals = [cell(n, t) for t in TASKS]
    if all(v is None for v in vals):
        lines.append(f"| {lab} | " + " | ".join("—" for _ in TASKS) + f" | — | not run | {rung_err(n)} |")
        continue
    avg = np.mean(vals) if all(v is not None for v in vals) else None
    d = (f"{DELTA.loc[n, 'avg_delta']:+.2f} [{DELTA.loc[n, 'avg_lo']:+.2f}, {DELTA.loc[n, 'avg_hi']:+.2f}]"
         if n in DELTA.index else "—")
    lines.append(f"| {lab} | " + " | ".join("—" if v is None else f"{v:.2f}" for v in vals)
                 + f" | {'—' if avg is None else f'{avg:.2f}'} | {d} | {rung_err(n)} |")
if E1:
    lines.append("")
    lines.append("| Mass estimation (Exp 1, median head) | " + " | ".join(TASKS) + " |")
    lines.append("|---|" + "---|" * len(TASKS))
    for k, lab in (("E_w", "E_logm, mass-weighted (cal)"), ("E_unw", "E_logm, unweighted (cal)"),
                   ("E_w_uncal", "E_logm, mass-weighted (uncal)"), ("bias_w", "bias on unread windows (cal)"),
                   ("unread", "share of held attention unread")):
        lines.append(f"| {lab} | " + " | ".join(f"{E1[t]['summary']['median head'][k]:.4f}" for t in TASKS) + " |")
    lines.append("| read-gate output error removed by completion, 1 - err(cal)/err(off) | " + " | ".join(
        f"{1 - E1[t]['rungs']['median head']['cal'] / E1[t]['rungs']['median head']['off']:.1%}" for t in TASKS) + " |")
md_table = "\n".join(lines)
keep_or_write(ROOT / "results" / "rebuttal_table.md", lambda p: p.write_text(md_table + f"\n\ncommit {SHA}\n"))
print(md_table)
status_table(STAGES[RUN])
print(f"\nAll results: {ROOT / 'results'}   commit {SHA[:10]}")

---
# Part II: Reviewer-2 follow-up

## 10. Follow-up setup: what is already on Drive, and what is new

The core run gave two answers of very different precision. Experiment 1's matched-state error reduction is tight; the
end-to-end LongBench deltas have CIs that include zero. Part II fills only what that left open, and never touches the core
run's data.

| Step (`FU_STEPS`) | Reviewer question | Jobs, only where Drive does not already hold them |
|---|---|---|
| `ceiling` | Is the ρ = 1.0 full read there for all four tasks, and how far from it is ρ = 0.25 with calibrated completion? | `ceiling_g1.0` |
| `stress` | Does completion matter more when fewer int2 windows are read? | `stress_g0.10_cal`, `stress_g0.10_off` |
| `exp1_stress` | The same question, on matched decode states | Experiment 1 at ρ = 0.10 |
| `scales` | How sensitive is accuracy to a common mass error? | `scale0.5`, `scale0.8`, `scale1.25`, `scale4` (`scale2` is from the core run) |
| `noise` | How sensitive is it to redistributed mass, with the total kept? | 2× and 4× σ̂_U at seeds 0, 1, 2 (1× seed 0 is from the core run) |
| `topup` | Can the CIs be tighter? | `FU_TOPUP_ARMS` extended to `FU_N_KEY` rows per task: **only the new rows are run** |

**Where things go.** Part II writes only under `DRIVE_ROOT/followup/`:

```
followup/
  manifest.json                             commit + SHA-256 of each task file (written once)
  longbench/<arm>/<task>.<a>-<b>.jsonl      predictions for rows a..b-1 of the task file
                  <task>.<a>-<b>.slice.json the row range, its ids, the task file's SHA-256, the arm's spec
                  <task>.<a>-<b>.meta.json  the runner's sidecar, written last: it is the done-marker
  exp1_g0.10/                               Experiment 1 at rho = 0.10
  logs/  refused/                           the follow-up's own: no file of the core run can be overwritten
  results/                                  sections 12-15, rebuilt from the files above on every run (they hold no data)
```

**Segments.** An arm's predictions on a task are a union of row ranges. The core run's file covers rows 0…N₀−1 and is read
where it is. The follow-up adds `<a>-<b>` files. Every segment is verified again on every load before any row of it is
scored, against everything below:
* everything section 5 checks: gate verdict, realised read fraction, operating point, completion spec and its runtime proof;
* its ids, which must be exactly rows a…b−1 of the task file, in order;
* its answers, its row count, and its commit;
* for follow-up segments, the task file's SHA-256.

A segment that fails is not scored, and a job never runs over it.

**How a top-up runs only the new rows.** The runner scores the first `num_samples` rows of `$LONGBENCH_LOCAL_DIR/<task>.jsonl`.
A top-up from N₀ to N₁ points it at a folder whose `<task>.jsonl` holds rows N₀…N₁−1 of the original file, byte for byte.
Nothing at the pinned commit changes.

A row's prediction does not depend on which rows ran before it:
* decoding is greedy;
* the cache is built fresh for every example;
* the noise arms hash their noise from (seed, layer, head, window id).

What remains is GPU nondeterminism, and the `cal_repeat` floor measures it.

In [ ]:
# ---------- Part II setup: the follow-up's own folders, its arm registry, row-range segments ----------
import hashlib, zlib

FU = ROOT / "followup"
FU_LB, FU_RES, FU_LOGS, FU_REF = FU / "longbench", FU / "results", FU / "logs", FU / "refused"
FU_EXP1 = FU / f"exp1_g{STRESS_GATE:.2f}"
for d in (FU_LB, FU_RES, FU_LOGS, FU_REF, FU_EXP1):
    d.mkdir(parents=True, exist_ok=True)

# The rows every segment is cut from, byte for byte. Split on b"\n" only: str.splitlines() would also split on a
# U+2028 inside a JSON string, and the runner's JSON reader does not.
DS_LINES, DS_IDS, DS_SHA = {}, {}, {}
for t in TASKS:
    raw = (LB_DIR / f"{t}.jsonl").read_bytes()
    DS_SHA[t] = hashlib.sha256(raw).hexdigest()
    DS_LINES[t] = [l for l in raw.split(b"\n") if l.strip()]
    DS_IDS[t] = [json.loads(l)["_id"] for l in DS_LINES[t]]

fu_manifest = FU / "manifest.json"
if fu_manifest.exists():
    fm = json.loads(fu_manifest.read_text())
    if fm["sha"] != SHA:
        raise RuntimeError(f"{FU} was started on commit {fm['sha'][:10]}; this clone is {SHA[:10]}.")
    moved = [t for t in TASKS if fm["dataset_sha256"].get(t, DS_SHA[t]) != DS_SHA[t]]
    if moved:
        raise RuntimeError(f"The LongBench files for {moved} are not the ones the follow-up started on. A top-up is "
                           "only valid against the rows its base ran on: restore them from THUDM/LongBench data.zip.")
else:
    fu_manifest.write_text(json.dumps({"sha": SHA, "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                                       "dataset_sha256": DS_SHA}, indent=2))

# Every arm of section 5 under its own name and spec (an arm is looked up by name, in both trees), plus the noise seeds
# Part II adds. An arm that was already run, in either tree, is reused as it is.
FU_ARMS = dict(ARMS)
def _fu_arm(name, exp, comp, sigma_mult=None):
    FU_ARMS.setdefault(name, dict(stage="followup", exp=exp, gate=GATE, comp=comp, sigma_mult=sigma_mult, floor=False))
for c in FU_SCALES:
    _fu_arm(f"scale{c:g}", "4A", C(scale=c))
for s in FU_NOISE_SEEDS:
    for m in (1, *FU_NOISE_MULTS):
        _fu_arm(f"noise{m}x_s{s}", "4B", C(seed=s), sigma_mult=m)
CEIL = "ceiling_g1.0"
STRESS = (f"stress_g{STRESS_GATE:.2f}_cal", f"stress_g{STRESS_GATE:.2f}_off")
assert all(n in FU_ARMS and FU_ARMS[n]["gate"] == g for n, g in ((CEIL, 1.0), (STRESS[0], STRESS_GATE),
                                                                  (STRESS[1], STRESS_GATE))), "section 5 changed"

_HEX = re.compile(r"[0-9a-f]{7,40}")

def _rows(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

def _seg_errors(meta, rows, arm, comp, t, a0, b0):
    """Everything a segment must satisfy to stand in for rows [a0, b0) of task t under this arm."""
    errs = verify(meta, arm, comp)    # gate verdict, read fraction, operating point, completion spec and its proof
    if [r.get("_id") for r in rows] != DS_IDS[t][a0:b0]:
        errs.append(f"its ids are not rows {a0}..{b0 - 1} of {t}.jsonl, in order")
    elif any(r.get("answers") != json.loads(l).get("answers") for r, l in zip(rows, DS_LINES[t][a0:b0])):
        errs.append("its answers differ from the dataset rows")
    if meta.get("num_examples") != len(rows):
        errs.append(f"num_examples = {meta.get('num_examples')}, but the file holds {len(rows)} rows")
    c = str(meta.get("commit_sha") or "")
    if _HEX.fullmatch(c) and not SHA.startswith(c):
        errs.append(f"it ran on commit {c[:10]}, and this folder holds {SHA[:10]}")
    return errs

def fu_segments(name, t):
    """(good, bad) segments of arm `name` on task `t`. A segment is a contiguous row range [a, b) of the task file
    with its own predictions and sidecar: the core run's `<task>.jsonl` (read in place, never moved) and the
    follow-up's `<task>.<a>-<b>.*`. Where two overlap, the core run's is kept. Reads only."""
    arm = FU_ARMS[name]
    comp = resolved_comp(arm)          # raises Blocked for the noise arms until sigma_hat.json exists
    cand = []
    o = ROOT / "longbench" / name
    if (o / f"{t}.meta.json").exists() and (o / f"{t}.jsonl").exists():
        rows, meta = _rows(o / f"{t}.jsonl"), json.loads((o / f"{t}.meta.json").read_text())
        cand.append(dict(src="core", a=0, b=len(rows), jsonl=o / f"{t}.jsonl", meta=meta,
                         errs=_seg_errors(meta, rows, arm, comp, t, 0, len(rows))))
    for mf in sorted((FU_LB / name).glob(f"{t}.*.meta.json")):
        m = re.fullmatch(rf"{re.escape(t)}\.(\d+)-(\d+)\.meta\.json", mf.name)
        if not m:
            continue
        stem = f"{t}.{m[1]}-{m[2]}"
        jf, sf = mf.with_name(f"{stem}.jsonl"), mf.with_name(f"{stem}.slice.json")
        seg = dict(src="followup", a=int(m[1]), b=int(m[2]), jsonl=jf, meta=json.loads(mf.read_text()), errs=[])
        if not (jf.exists() and sf.exists()):
            seg["errs"].append("its predictions or slice record are missing")
        else:
            seg["errs"] = _seg_errors(seg["meta"], _rows(jf), arm, comp, t, seg["a"], seg["b"])
            if json.loads(sf.read_text()).get("dataset_sha256") != DS_SHA[t]:
                seg["errs"].append("it was cut from a different copy of the task file")
        cand.append(seg)
    good, bad = [], []
    for s in cand:
        if not s["errs"] and any(s["a"] < g["b"] and g["a"] < s["b"] for g in good):
            s["errs"].append("it overlaps rows another segment already covers")
        (bad if s["errs"] else good).append(s)
    return sorted(good, key=lambda s: s["a"]), bad

def _prefix(segs):
    """k such that rows 0..k-1 are covered."""
    k = 0
    for s in sorted(segs, key=lambda s: s["a"]):
        if s["a"] > k:
            break
        k = max(k, s["b"])
    return k

def _span(segs):
    return " + ".join(f"{s['src']} {s['a']}-{s['b'] - 1}" for s in segs) or "none"

# The core N, read from the files rather than from NUM_SAMPLES, and the measured speed every projection uses.
N0, EPS = {}, {}
for t in TASKS:
    good, bad = fu_segments("cal", t)
    core = [s for s in good if s["src"] == "core"]
    if not core:
        raise RuntimeError(f"The core run's cal arm has no verified {t} file on Drive"
                           + (" (" + "; ".join(bad[0]["errs"]) + ")" if bad else "")
                           + ". Part II pairs every arm with it, so it cannot start.")
    N0[t], EPS[t] = core[0]["b"], core[0]["meta"].get("examples_per_second")
N_SENS = {t: min(FU_N_SENS or N0[t], len(DS_IDS[t])) for t in TASKS}
N_KEY = {t: max(N_SENS[t], min(FU_N_KEY or 0, len(DS_IDS[t]))) for t in TASKS}
print("core run, cal rows per task:", N0)
print("new arms at (FU_N_SENS):    ", N_SENS)
print("top-up target (FU_N_KEY):   ", N_KEY if FU_N_KEY else "none")
if any(N0[t] != NUM_SAMPLES for t in TASKS):
    print(f"\nnote: NUM_SAMPLES = {NUM_SAMPLES}, but the core arms ran at the N above. Sections 7-9 check sidecars "
          "against NUM_SAMPLES and will not score them; set NUM_SAMPLES to the core N for those sections. Part II "
          "reads N from the files.")

## 11. Follow-up jobs: plan, projected time, run

**Order.** The order is also the priority:
1. `control`, which appears only if `FU_N_SENS` is set above the core N: `cal` and `off` must cover the new rows before anything
   is paired with them;
2. the steps, in `FU_STEPS` order;
3. inside `noise`, seed-major: every level gets seed 0 before any level gets seed 1.

One process runs one arm over all four tasks, so the model loads once per arm. After a disconnect, run everything again:
finished segments are kept, and a job runs only the rows still missing. `FU_RUN_JOBS = False` prints the plan and its
projected time, and runs nothing.

**Time.** The projection is arithmetic:
* the core `cal` arm's **measured** examples/second per task;
* +20% on epilogue arms;
* +90 s per job for the model load.

At 40 rows/task, one arm takes roughly 25–35 min. The default plan is about 9–12 h on one A100, which is why the order matters:
* 13 new arms;
* the four Experiment 1 collector runs at ρ = 0.10;
* five top-ups of 60 rows/task.

In the table, `✓ 40` means rows 0–39 are verified on Drive, and `40→100` means rows 40–99 are still to run.

In [ ]:
def fu_plan():
    """Every follow-up job, in run order. An lb job's `target` is the number of rows per task it must cover."""
    jobs = []
    def lb(step, name, target):
        jobs.append(dict(step=step, kind="lb", name=name, target=dict(target)))
    if any(N_SENS[t] > N0[t] for t in TASKS):      # FU_N_SENS above the core N: the controls must cover it first
        lb("control", "cal", N_SENS)
        lb("control", "off", N_SENS)
    for step in FU_STEPS:
        if step == "ceiling":
            lb(step, CEIL, N_SENS)
        elif step == "stress":
            for n in STRESS:
                lb(step, n, N_SENS)
        elif step == "exp1_stress":
            jobs.append(dict(step=step, kind="exp1", name=FU_EXP1.name, target=None))
        elif step == "scales":
            for c in FU_SCALES:
                lb(step, f"scale{c:g}", N_SENS)
        elif step == "noise":
            for s in FU_NOISE_SEEDS:
                for m in ((1,) if FU_NOISE_1X_SEEDS else ()) + tuple(FU_NOISE_MULTS):
                    lb(step, f"noise{m}x_s{s}", N_SENS)
        elif step == "topup":
            for n in (FU_TOPUP_ARMS if FU_N_KEY else ()):
                lb(step, n, N_KEY)
        else:
            raise ValueError(f"unknown step {step!r} in FU_STEPS")
    return jobs

def fu_missing(job):
    """{task: (a, b)}: the rows an lb job still has to run. Blocked if an invalid segment is in the way."""
    todo = {}
    for t, n in job["target"].items():
        good, bad = fu_segments(job["name"], t)
        if bad:
            raise Blocked(f"{job['name']}/{t}: a segment fails verification ("
                          + "; ".join(f"{s['src']} {s['a']}-{s['b'] - 1}: " + ", ".join(s["errs"]) for s in bad)
                          + f"). Part II never runs over it and never moves core files. Inspect it; to rerun a "
                          f"follow-up segment's rows, move its three files to {FU_REF}.")
        k = _prefix(good)
        if k < n:
            todo[t] = (k, min([n] + [s["a"] for s in good if s["a"] > k]))
    return todo

def fu_exp1_missing():
    return [t for t in TASKS if not (FU_EXP1 / f"{t}_results.json").exists()]

def fu_minutes(job, todo):
    if not all(EPS.get(t) for t in todo):
        return None
    epi = 1.0 if FU_ARMS[job["name"]]["comp"]["mode"] == "native" else 1.2
    return (sum((b - a) / EPS[t] for t, (a, b) in todo.items()) * epi + 90) / 60

def fu_status(jobs=None):
    jobs = fu_plan() if jobs is None else jobs
    total, exp1_left, sim = 0.0, False, {}   # sim: rows an arm will cover once the jobs above it have run
    print(f"{'step':13s}{'job':20s}{'state':>9s}" + "".join(f"{t:>14s}" for t in TASKS) + f"{'~min':>7s}")
    for j in jobs:
        mins = None
        try:
            if j["kind"] == "exp1":
                miss = fu_exp1_missing()
                cells = ["to run" if t in miss else "✓" for t in TASKS]
                state, exp1_left = ("pending" if miss else "done"), exp1_left or bool(miss)
            else:
                todo = fu_missing(j)
                adj = {t: (max(a, sim.get((j["name"], t), 0)), b) for t, (a, b) in todo.items()}
                adj = {t: ab for t, ab in adj.items() if ab[0] < ab[1]}
                for t, n in j["target"].items():
                    sim[(j["name"], t)] = max(sim.get((j["name"], t), 0), n)
                cells = [f"{adj[t][0]}→{adj[t][1]}" if t in adj else "earlier job" if t in todo
                         else f"✓ {j['target'][t]}" for t in TASKS]
                state = "pending" if adj else "planned" if todo else "done"
                if adj:
                    mins = fu_minutes(j, adj)
                    total += mins or 0.0
        except Blocked as e:
            cells, state = ["-"] * len(TASKS), "BLOCKED"
            print(f"   BLOCKED {j['name']}: {e}")
        print(f"{j['step']:13s}{j['name']:20s}{state:>9s}" + "".join(f"{c:>14s}" for c in cells)
              + (f"{mins:7.0f}" if mins else ""))
    print(f"\nprojected remaining: {total / 60:.1f} h of LongBench (the core cal arm's measured ex/s per task, +20% on "
          "epilogue arms, +90 s load per job)"
          + ("\n  + Experiment 1 at rho = 0.10: not projected, because the core run did not time its collector runs"
             if exp1_left else ""))
    return jobs

FU_JOBS = fu_status()

In [ ]:
def fu_lb_job(job):
    """Run only the missing rows of one arm, all tasks in one process, then verify each task's segment and copy it to
    Drive. A finished segment is never overwritten; a refused one goes to followup/refused/."""
    name, arm = job["name"], FU_ARMS[job["name"]]
    comp = resolved_comp(arm)
    todo = fu_missing(job)
    if not todo:
        return "done"
    key = hashlib.sha1(json.dumps(sorted(todo.items())).encode()).hexdigest()[:8]
    sdir, local = RUNS / "fu_slices" / name / key, RUNS / "fu_longbench" / name / key
    for d in (sdir, local):
        shutil.rmtree(d, ignore_errors=True)
        d.mkdir(parents=True)
    for t, (a, b) in todo.items():                  # rows a..b-1, byte for byte; the runner takes the first rows
        (sdir / f"{t}.jsonl").write_bytes(b"\n".join(DS_LINES[t][a:b]) + b"\n")
    n_req = max(b - a for a, b in todo.values())
    print(f"{name}: " + ", ".join(f"{t} rows {a}-{b - 1}" for t, (a, b) in todo.items()))
    cmd = [PY, "main.py", "--config", "configs/longbench_ours_flash_attn.yaml", "--override",
           f"longbench.datasets=[{','.join(todo)}]", f"longbench.output_dir={local}",
           f"longbench.num_samples={n_req}", "longbench.resume=false",
           f"longbench.skip_oom={str(SKIP_OOM).lower()}", *overrides(arm, comp)]
    failure, refused = None, []
    try:
        run(cmd, f"{name}.{key}", env=dict(ENV, LONGBENCH_LOCAL_DIR=str(sdir)), log_dir=FU_LOGS)
    except RuntimeError as e:
        failure = e
    dst = FU_LB / name
    dst.mkdir(parents=True, exist_ok=True)
    for t, (a, b) in todo.items():
        mf, pf = local / f"{t}.meta.json", local / f"{t}.jsonl"
        if not (mf.exists() and pf.exists()):
            continue
        meta = json.loads(mf.read_text())
        errs = _seg_errors(meta, _rows(pf), arm, comp, t, a, b)
        if meta.get("num_samples_requested") != n_req:
            errs.append(f"num_samples_requested = {meta.get('num_samples_requested')!r}, expected {n_req}")
        stem = f"{t}.{a:04d}-{b:04d}"
        if errs:
            try:
                _refuse([pf, mf], f"{name}_{stem}_{key}", errs, base=FU_REF)
            except RuntimeError as e:
                refused.append(str(e))
            continue
        if (dst / f"{stem}.meta.json").exists():
            raise RuntimeError(f"{dst / stem}.* is already finished; refusing to overwrite it")
        shutil.copy(pf, dst / f"{stem}.jsonl")
        (dst / f"{stem}.slice.json").write_text(json.dumps({
            "arm": name, "task": t, "rows": [a, b], "ids": DS_IDS[t][a:b], "dataset_sha256": DS_SHA[t],
            "slice_sha256": hashlib.sha256((sdir / f"{t}.jsonl").read_bytes()).hexdigest(),
            "spec": {"gate": arm["gate"], **comp}, "num_samples_requested": n_req, "sha": SHA,
            "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}, indent=1))
        shutil.copy(mf, dst / f"{stem}.meta.json")      # last: the sidecar is the done-marker
    if refused:
        raise RuntimeError("\n".join(refused))
    if failure is not None:
        raise failure
    return "ran"

def fu_exp1_job(job):
    """Experiment 1 at rho = STRESS_GATE: section 6's job with its own gate and folders (finished tasks are kept)."""
    if not fu_exp1_missing():
        return "done"
    for t in TASKS:
        exp1_job(t, gate=STRESS_GATE, out_dir=FU_EXP1, tag=f"fu_{FU_EXP1.name}", log_dir=FU_LOGS, refused=FU_REF)
    return "ran"

def fu_run(jobs):
    t0 = time.time()
    for j in jobs:
        try:
            r = fu_lb_job(j) if j["kind"] == "lb" else fu_exp1_job(j)
        except Blocked as e:
            print(f"BLOCKED {j['name']}: {e}")
            continue
        if r == "ran":
            print(f"    {j['step']}/{j['name']} done, {(time.time() - t0) / 3600:.1f} h into this session")
    fu_status(jobs)

if FU_RUN_JOBS:
    fu_run(FU_JOBS)
else:
    print("FU_RUN_JOBS = False: nothing was run.")

## 12. ρ = 1.0: is the full read available, and how close is calibrated completion at ρ = 0.25?

`ceiling_g1.0` reads every int2 window (`quant_gate_ratio = 1.0`) through the same gate code. Nothing is left unread, so the
completion term has nothing to stand in for. It is the gate's control arm, and it holds the same windows, so memory is unchanged.

* **Availability**, per task: rows covered, where they come from, the gate verdict, and the realised read fraction (must be 1.0).
* **Comparison**: task scores on the rows that `ceiling_g1.0`, `cal` and `off` all share, and paired deltas with 95% bootstrap
  CIs.
* **Share of the read-gate gap that completion closes**: (cal − off) / (ρ1.0 − off). It is defined only when ρ = 1.0 beats
  `off` significantly.

The first cell loads and scores every verified segment of every arm once. Sections 13–15 reuse it. The scoring rule is section
8's: the repo's own LongBench metrics and first-line rule, null (OOM) = 0 in task scores, and paired deltas over rows non-null
in both arms. Each comparison bootstraps from its own fixed seed, so a number does not move when other arms are added.

In [ ]:
os.chdir(REPO)
sys.path.insert(0, str(REPO))
from modules.evaluation.longbench_scoring import METRIC_FN_REGISTRY, _FIRST_LINE_DATASETS
D2M = json.loads((REPO / "data/longbench_configs/dataset2metric.json").read_text())

def _score_rows(rows, task):
    """{_id: score or None}, {_id: pred}: section 8's rule."""
    fn, S, P = METRIC_FN_REGISTRY[D2M[task]], {}, {}
    for ex in rows:
        pred = P[ex["_id"]] = ex.get("pred")
        if pred is None:
            S[ex["_id"]] = None
            continue
        if task in _FIRST_LINE_DATASETS:
            pred = pred.lstrip("\n").split("\n")[0]
        S[ex["_id"]] = 100.0 * max((fn(pred, gt, all_classes=ex.get("all_classes") or [])
                                    for gt in ex.get("answers", [])), default=0.0)
    return S, P

# Every verified segment of every arm, scored once: (arm, task) -> ({_id: score}, {_id: pred}).
FS, SEGS = {}, {}
for n in FU_ARMS:
    for t in TASKS:
        try:
            good, bad = fu_segments(n, t)
        except Blocked:
            continue
        for s in bad:
            print(f"NOT SCORED {n}/{t} {s['src']} rows {s['a']}-{s['b'] - 1}: " + "; ".join(s["errs"]))
        if good:
            S, P = {}, {}
            for s in good:
                sc, pr = _score_rows(_rows(s["jsonl"]), t)
                S.update(sc)
                P.update(pr)
            FS[(n, t)], SEGS[(n, t)] = (S, P), good
print(f"scored {len({n for n, _ in FS})} arms: " + ", ".join(sorted({n for n, _ in FS})))

def _rng(*key):
    return np.random.default_rng(zlib.crc32("|".join(map(str, key)).encode()))

def have(*names):
    return all((n, t) in FS for n in names for t in TASKS)

def _common(names, t):
    """Rows, in dataset order, that every arm in `names` scored non-null."""
    Ss = [FS[(n, t)][0] for n in names]
    return [i for i in DS_IDS[t] if all(S.get(i) is not None for S in Ss)]

def _arrays(names, t):
    ids = _common(names, t)
    return ids, {n: np.array([FS[(n, t)][0][i] for i in ids], dtype=float) for n in names}

def dstats(names, f, key):
    """Paired statistic over the rows every arm in `names` shares. f(A) gives the per-row delta for one task, with
    A = {arm: scores}. Returns the per-task means with 95% CIs; the 4-task average with a 95% CI (tasks resampled
    independently and averaged, as in section 8); the MDE at 80% power (two-sided 5%); n; and whether the CI
    excludes 0. Returns None if any arm lacks a task."""
    if not have(*names):
        return None
    out, dists, means, ses, n = {"per_task": {}}, [], [], [], 0
    for t in TASKS:
        ids, A = _arrays(names, t)
        if len(ids) < 2:
            return None
        d = f(A)
        bd = d[_rng(*key, t).integers(0, len(d), (BOOTSTRAP_ITERS, len(d)))].mean(1)
        lo, hi = np.percentile(bd, [2.5, 97.5])
        out["per_task"][t] = (float(d.mean()), float(lo), float(hi), len(d))
        dists.append(bd); means.append(d.mean()); ses.append(d.std(ddof=1) / math.sqrt(len(d))); n += len(d)
    avg = np.mean(dists, 0)
    out.update(delta=float(np.mean(means)), lo=float(np.percentile(avg, 2.5)), hi=float(np.percentile(avg, 97.5)),
               mde80=float(2.80 * math.sqrt(sum(s * s for s in ses)) / len(TASKS)), n=n)
    out["sig"] = out["lo"] > 0 or out["hi"] < 0
    return out

def delta(x, y):
    """x - y, paired over the rows both scored."""
    return dstats((x, y), lambda A: A[x] - A[y], ("delta", x, y))

def ratio_stats(names, fnum, fden, key):
    """Ratio of two 4-task average deltas, numerator and denominator resampled on the same rows."""
    if not have(*names):
        return None
    nums, dens, pn, pdn = [], [], [], []
    for t in TASKS:
        ids, A = _arrays(names, t)
        if len(ids) < 2:
            return None
        idx = _rng(*key, t).integers(0, len(ids), (BOOTSTRAP_ITERS, len(ids)))
        a, b = fnum(A), fden(A)
        nums.append(a[idx].mean(1)); dens.append(b[idx].mean(1)); pn.append(a.mean()); pdn.append(b.mean())
    N, D = np.mean(nums, 0), np.mean(dens, 0)
    with np.errstate(divide="ignore", invalid="ignore"):
        R = N / D
    return dict(point=float(np.mean(pn) / np.mean(pdn)) if np.mean(pdn) else float("nan"),
                lo=float(np.percentile(R, 2.5)), hi=float(np.percentile(R, 97.5)),
                den_lo=float(np.percentile(D, 2.5)))

def task_score(n, t, ids=None):
    """THUDM convention: the mean over rows, a null (OOM) prediction scoring 0."""
    S = FS[(n, t)][0]
    return float(np.mean([S[i] or 0.0 for i in (S if ids is None else ids)]))

def changed(x, y):
    """Share of shared rows whose prediction text differs at all."""
    same = [FS[(x, t)][1][i] == FS[(y, t)][1][i] for t in TASKS for i in _common((x, y), t)]
    return 1.0 - float(np.mean(same)) if same else float("nan")

def ci(r, nd=2):
    return "not run" if r is None else f"{r['delta']:+.{nd}f} [{r['lo']:+.{nd}f}, {r['hi']:+.{nd}f}]"

def dtable(rows):
    """[(label, dstats)] -> per-task and average deltas with CIs, MDE, n, significance."""
    out = {}
    for lab, r in rows:
        if r is None:
            out[lab] = {"average": "not run"}
            continue
        out[lab] = {**{t: f"{m:+.2f} [{lo:+.2f},{hi:+.2f}]" for t, (m, lo, hi, _) in r["per_task"].items()},
                    "average": ci(r), "MDE80": round(r["mde80"], 2), "n": r["n"],
                    "CI excl. 0": "yes" if r["sig"] else "no"}
    return pd.DataFrame(out).T

In [ ]:
print(f"Availability of the rho = 1.0 full read ({CEIL}) on Drive, target {N_SENS} rows/task:\n")
av = {}
for t in TASKS:
    good, bad = fu_segments(CEIL, t)
    rg = [s["meta"].get("read_gate") or {} for s in good]
    rf = [r.get("read_fraction") for r in rg if r.get("read_fraction") is not None]
    av[t] = {"rows": _prefix(good), "segments": _span(good),
             "verdict": ",".join(sorted({str(r.get("verdict")) for r in rg})) or "-",
             "read_fraction": f"{min(rf):.4f}" if rf else "-",
             "gpu": ",".join(sorted({str(s["meta"].get("gpu_name")) for s in good})) or "-",
             "invalid": "; ".join(f"{s['src']} {s['a']}-{s['b'] - 1}: " + ", ".join(s["errs"]) for s in bad)}
AV = pd.DataFrame(av).T
print(AV.to_string())
AV.to_csv(FU_RES / "rho1_availability.csv")
short = [t for t in TASKS if av[t]["rows"] < N_SENS[t]]
if short:
    print(f"\nNOT COMPLETE: {CEIL} is missing or short on {short}. Section 11 runs only those rows.")
else:
    print(f"\nAVAILABLE: {CEIL} is on Drive for all four tasks, and every segment verifies: gate verdict "
          "'gated', realised read fraction 1.0, operating point, native completion with completion_applied = 0, "
          "the dataset's own rows in order.")

if have(CEIL, "cal", "off"):
    ids = {t: _common((CEIL, "cal", "off"), t) for t in TASKS}
    T = pd.DataFrame({t: {n: task_score(n, t, ids[t]) for n in (CEIL, "cal", "off")} for t in TASKS})
    T["average"] = T[TASKS].mean(axis=1)
    print("\nTask scores on the rows all three share (" + ", ".join(f"{t} {len(ids[t])}" for t in TASKS) + "):")
    print(T.round(2).to_string())
    R1 = dtable([(f"rho 1.00 - cal@{GATE:.2f}", delta(CEIL, "cal")),
                 (f"off@{GATE:.2f} - cal@{GATE:.2f}", delta("off", "cal")),
                 (f"rho 1.00 - off@{GATE:.2f}", delta(CEIL, "off"))])
    print("\nPaired deltas, points [95% CI]:")
    print(R1.to_string())
    gap = ratio_stats((CEIL, "cal", "off"), lambda A: A["cal"] - A["off"], lambda A: A[CEIL] - A["off"], ("gap",))
    if gap and gap["den_lo"] > 0:
        print(f"\nShare of the read-gate gap closed by completion, (cal - off)/(rho1.0 - off): {gap['point']:.2f} "
              f"[{gap['lo']:.2f}, {gap['hi']:.2f}]")
    else:
        print("\nShare of the read-gate gap closed by completion: undefined, because rho = 1.0 does not beat 'off' "
              "significantly at this N (the denominator's CI includes 0).")
    T.to_csv(FU_RES / "rho1_task_scores.csv")
    R1.to_csv(FU_RES / "rho1_deltas.csv")

## 13. ρ = 0.10 stress test: does completion matter more when fewer windows are read?

At ρ = 0.10 the gate opens 10% of the int2 windows rather than 25%, so the completion term stands in for more of the held
attention. Two measurements, read side by side:

* **Matched states**, Experiment 1 at ρ = 0.10 against the core run's ρ = 0.25:
  * the unread share of held attention;
  * the attention-output error with and without completion;
  * the share **and** the absolute amount of that error that completion removes.

  Both ρ use the same prompts, and each runs on its own decode trajectory.
* **End to end**, on LongBench:
  * the benefit of completion, `cal − off`, at each ρ;
  * the **difference-in-differences** `(cal − off)@0.10 − (cal − off)@0.25`, on the rows all four arms share, resampled
    jointly.

  A positive DiD whose CI excludes 0 is the end-to-end evidence that completion matters more at 0.10. A DiD whose CI includes 0
  means this N cannot tell.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#52514e",
                     "axes.labelcolor": "#0b0b0b", "xtick.color": "#52514e", "ytick.color": "#52514e",
                     "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.8,
                     "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb"})
BLUE, ORANGE, MUTED, BAND = "#2a78d6", "#eb6834", "#52514e", "#e6e5e1"
rho_s = lambda rho: f"{rho:.2f}"                 # 0.25, 0.10, 1.00: read ratios print alike everywhere

def _ref(ax, x, label, side="left", y=1):     # as in section 8's figure
    ax.axvline(x, color=MUTED, ls="--", lw=1, zorder=1)
    ax.annotate(label, (x, y), xycoords=("data", "axes fraction"), textcoords="offset points",
                xytext=(-4 if side == "left" else 4, -4 if y else 4), ha="right" if side == "left" else "left",
                va="top" if y else "bottom", color=MUTED, fontsize=9)

def _load_e1(d):
    return {t: json.loads((d / f"{t}_results.json").read_text()) for t in TASKS if (d / f"{t}_results.json").exists()}
E1C, E1S = _load_e1(ROOT / "exp1"), _load_e1(FU_EXP1)     # the core run's rho = GATE, and the follow-up's rho = 0.10

def rung(E, r, col="median head"):
    """Experiment 1's output error for variant r: the median over tasks of the median head."""
    v = [E[t]["rungs"][col].get(r) for t in E]
    v = [x for x in v if x is not None]
    return float(np.median(v)) if v else None

def _ms(E, rho):
    out = {}
    for t in E:
        m, s = E[t]["rungs"]["median head"], E[t]["summary"]["median head"]
        out[(rho, t)] = {"head_steps": E[t]["head_steps"], "unread": s["unread"], "err_off": m["off"],
                         "err_cal": m["cal"], "removed": 1 - m["cal"] / m["off"], "removed_abs": m["off"] - m["cal"],
                         "mass_oracle": m["mass_oracle"], "E_w": s["E_w"]}
    return out
MS = pd.DataFrame({**_ms(E1C, GATE), **_ms(E1S, STRESS_GATE)}).T
if len(MS):
    MS.index.names = ["rho", "task"]
    print("Matched states (Experiment 1): median head per task. err_* = rel. L2 output error vs o_held; removed = "
          "1 - err_cal/err_off; removed_abs = err_off - err_cal")
    print(MS.round(4).to_string())
    MS.to_csv(FU_RES / "stress_matched_states.csv")
MED = {rho: MS.xs(rho).median() for rho in (GATE, STRESS_GATE) if len(MS) and rho in MS.index.get_level_values(0)}
if len(MED) == 2:
    a, b = MED[GATE], MED[STRESS_GATE]
    print(f"\nmedian over tasks, rho {rho_s(GATE)} -> {rho_s(STRESS_GATE)}:")
    print(f"  unread share of held attention   {a['unread']:.1%} -> {b['unread']:.1%}")
    print(f"  output error without completion  {a['err_off']:.4f} -> {b['err_off']:.4f}")
    print(f"  output error with completion     {a['err_cal']:.4f} -> {b['err_cal']:.4f}")
    print(f"  share removed by completion      {a['removed']:.1%} -> {b['removed']:.1%}")
    print(f"  error removed by completion      {a['removed_abs']:.4f} -> {b['removed_abs']:.4f}  "
          f"({b['removed_abs'] / a['removed_abs']:.2f}x)")
elif STRESS_GATE not in MED:
    print(f"Experiment 1 at rho = {rho_s(STRESS_GATE)} is not on Drive yet (step 'exp1_stress').")

SC, SO = STRESS
B25 = dstats(("cal", "off"), lambda A: A["cal"] - A["off"], ("benefit", GATE))
B10 = dstats((SC, SO), lambda A: A[SC] - A[SO], ("benefit", STRESS_GATE))
DID = dstats((SC, SO, "cal", "off"), lambda A: (A[SC] - A[SO]) - (A["cal"] - A["off"]), ("did",))
g, s = rho_s(GATE), rho_s(STRESS_GATE)
ST = dtable([(f"cal - off @ {g}", B25), (f"cal - off @ {s}", B10), (f"DiD: (cal-off)@{s} - (cal-off)@{g}", DID),
             (f"cal@{s} - cal@{g}", delta(SC, "cal")), (f"off@{s} - off@{g}", delta(SO, "off"))])
print("\nEnd to end (LongBench), paired, points [95% CI]:")
print(ST.to_string())
ST.to_csv(FU_RES / "stress_deltas.csv")
if DID is not None:
    print(f"\nDiD {ci(DID)}: " + (f"the end-to-end benefit of completion is larger at rho = {s}." if DID["lo"] > 0 else
          f"the benefit is smaller at rho = {s}." if DID["hi"] < 0 else
          f"this N cannot tell whether the end-to-end benefit changes (MDE80 {DID['mde80']:.2f} points)."))

# Figure: matched-state error vs rho (left), end-to-end benefit vs rho (right).
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
ax = axes[0]
if len(MED) == 2:
    xs = [0, 1, 2]                               # categorical: 0.10, 0.25, 1.0
    for key, col, lab in (("err_off", ORANGE, "no completion (off)"), ("err_cal", BLUE, "calibrated (cal)")):
        ys = [MED[STRESS_GATE][key], MED[GATE][key], 0.0]
        ax.plot(xs, ys, color=col, lw=2, marker="o", ms=8, zorder=3, label=lab)
        ax.annotate(lab.split(" (")[1][:-1], (0, ys[0]), textcoords="offset points", xytext=(-10, 0), ha="right",
                    va="center", color=MUTED, fontsize=9)
    ax.set_xticks(xs, [rho_s(STRESS_GATE), rho_s(GATE), rho_s(1.0)])
    ax.set_xlim(-0.7, 2.3)
    ax.legend(frameon=False, loc="upper right", fontsize=9)
ax.set_xlabel("read ratio ρ  (1.0: nothing unread, 0 by construction)")
ax.set_ylabel("output error vs o_held (median head, median task)")
ax.set_title("Matched states: error the completion term stands in for", loc="left")
ax = axes[1]
ax.axhline(0, color=MUTED, lw=1, zorder=1)
fl = [abs(r["delta"]) for r in (delta("cal_repeat", "cal"), delta("cal_ep", "cal")) if r]
if fl:
    ax.axhspan(-max(fl), max(fl), color=BAND, lw=0)
pts = [(i, r) for i, r in ((0, B10), (1, B25)) if r]
if pts:
    ax.errorbar([i for i, _ in pts], [r["delta"] for _, r in pts],
                yerr=[[r["delta"] - r["lo"] for _, r in pts], [r["hi"] - r["delta"] for _, r in pts]],
                color=BLUE, lw=0, elinewidth=2, marker="o", ms=8, capsize=3, zorder=3)
ax.set_xticks([0, 1], [rho_s(STRESS_GATE), rho_s(GATE)])
ax.set_xlim(-0.6, 1.6)
ax.set_xlabel("read ratio ρ")
ax.set_ylabel("cal − off, LongBench 4-task average (points)")
ax.set_title("End to end: benefit of completion", loc="left")
fig.text(0.01, -0.03, "Right: 95% bootstrap CIs, paired rows; grey band = largest |Δ| of the floors cal_repeat and "
         "cal_ep vs cal.", color=MUTED, fontsize=9)
fig.tight_layout()
fig.savefig(FU_RES / "stress.png", dpi=200, bbox_inches="tight")
plt.show()

## 14. End-to-end sensitivity: common mass scale `c`, and redistribution noise at 1×, 2× and 4× σ̂_U over three seeds

Every arm is paired with `cal` on the rows they share.
* `scale c`: every unread window's mass is multiplied by c, so the term is over- or under-credited by the same factor
  everywhere. `c = 1` is `cal`, and `off` is `c = 0`.
* `noise k×`: each unread window's mass is multiplied by `exp(σ·ε)`, with σ = k·σ̂_U, and the total is then restored exactly.
  It is a pure redistribution, orthogonal to `c`, and each seed draws different noise.
  * A level's effect is the **seed mean**, `mean_s(arm_s − cal)` per row, with a 95% CI from resampling rows.
  * The spread across seeds is shown next to it.
* The matched-state column is Experiment 1's attention-output error under the same perturbation (median head, median task):
  * scales are read directly;
  * noise levels are interpolated between the σ rungs (0.25, 0.5, 1, 2 nats) and marked `~`, or `>2` beyond the last rung.
* `changed` is the share of rows whose answer text differs from `cal`'s at all.
* Read every Δ against the two floors: `cal_repeat` (run-to-run) and `cal_ep` (the epilogue at identity).

In [ ]:
SH = json.loads(SIGMA_FILE.read_text()) if SIGMA_FILE.exists() else {}
SIG = SH.get("sigma_hat_U")
SCALES = sorted({*FU_SCALES, 2.0})
NOISE_LEVELS = (1, *FU_NOISE_MULTS)

def noise_rung(sig, E=None, col="median head"):
    """Exp 1 output error at sigma nats, interpolated between the rungs; None beyond the last one."""
    E = E1C if E is None else E
    xs, rs = [0.0, 0.25, 0.5, 1.0, 2.0], ["cal", "noise0.25", "noise0.5", "noise1", "noise2"]
    ys = [rung(E, r, col) for r in rs]
    if sig is None or any(y is None for y in ys) or sig > xs[-1]:
        return None
    return float(np.interp(sig, xs, ys))

def seed_mean(m):
    """The seed-mean delta vs cal at noise level m: (stats, per-seed stats, the seeds present)."""
    names = [f"noise{m}x_s{s}" for s in FU_NOISE_SEEDS if have(f"noise{m}x_s{s}")]
    if not names or not have("cal"):
        return None, {}, names
    r = dstats(("cal", *names), lambda A: np.mean([A[n] - A["cal"] for n in names], 0), ("noise", m, *names))
    return r, {n: delta(n, "cal") for n in names}, names

def _fmt_ms(v, approx=False):
    return "—" if v is None else ("~" if approx else "") + f"{v:.4f}"

rows = {}
for name, lab in (("off", "off (c = 0)"), ("cal_repeat", "floor: cal_repeat"), ("cal_ep", "floor: cal_ep")):
    r = delta(name, "cal")
    rows[lab] = dict(r=r, changed=changed(name, "cal") if r else None,
                     ms=_fmt_ms(rung(E1C, "off")) if name == "off" else "—")
for c in SCALES:
    n = f"scale{c:g}"
    r = delta(n, "cal")
    rows[f"scale c = {c:g}"] = dict(r=r, changed=changed(n, "cal") if r else None, ms=_fmt_ms(rung(E1C, n)))
SEEDS = {}
for m in NOISE_LEVELS:
    r, per, names = seed_mean(m)
    SEEDS[m] = (r, per)
    sig = None if SIG is None else m * SIG
    ms = noise_rung(sig)
    ms = ">2 nats" if sig is not None and sig > 2.0 else _fmt_ms(ms, approx=True)
    if len(names) == 1:
        rows[f"noise {m}x, seed {names[0].rsplit('_s', 1)[1]} (one seed)"] = dict(r=r, changed=changed(names[0], "cal"), ms=ms)
        continue
    for n, pr in per.items():
        rows[f"noise {m}x, seed {n.rsplit('_s', 1)[1]}"] = dict(r=pr, changed=changed(n, "cal") if pr else None, ms="")
    spread = [p["delta"] for p in per.values() if p]
    rows[f"noise {m}x, mean of {len(names)} seeds"] = dict(
        r=r, changed=None, spread=f"{min(spread):+.2f} .. {max(spread):+.2f}" if spread else "", ms=ms)
SENS = dtable([(k, v["r"]) for k, v in rows.items()])
SENS["changed"] = [("—" if v.get("changed") is None or np.isnan(v["changed"]) else f"{v['changed']:.0%}")
                   for v in rows.values()]
SENS["seed range"] = [v.get("spread", "") for v in rows.values()]
SENS["matched-state err"] = [v["ms"] for v in rows.values()]
print(f"Δ vs cal, points [95% CI], paired rows. sigma_hat_U = {SIG if SIG is None else round(SIG, 3)} nats; the "
      f"matched-state error of cal itself is {_fmt_ms(rung(E1C, 'cal'))}.")
print(SENS.to_string())
SENS.to_csv(FU_RES / "sensitivity_deltas.csv")
sens_rows = [v["r"] for k, v in rows.items()
             if v["r"] and (k.startswith("scale") or (k.startswith("noise") and ("mean" in k or "one seed" in k)))]
n_sig = sum(r["sig"] for r in sens_rows)
print(f"\n{n_sig} of {len(sens_rows)} sensitivity points (scales, noise seed-means) have a 95% CI that excludes 0; "
      f"about {0.05 * len(sens_rows):.1f} would by chance alone if nothing moved.")

# Figure: end to end (top) and matched states (bottom), the same perturbations.
fl = [abs(rows[k]["r"]["delta"]) for k in ("floor: cal_repeat", "floor: cal_ep") if rows[k]["r"]]
floor = max(fl) if fl else 0.0
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
ax = axes[0, 0]
ax.axhline(0, color=MUTED, lw=1, zorder=1)
ax.axhspan(-floor, floor, color=BAND, lw=0)
pts = [(0.0, dict(delta=0.0, lo=0.0, hi=0.0))] + [(np.log2(c), rows[f"scale c = {c:g}"]["r"]) for c in SCALES
                                                  if rows[f"scale c = {c:g}"]["r"]]
pts.sort(key=lambda p: p[0])
ax.errorbar([x for x, _ in pts], [r["delta"] for _, r in pts], yerr=[[r["delta"] - r["lo"] for _, r in pts],
            [r["hi"] - r["delta"] for _, r in pts]], color=BLUE, lw=2, marker="o", ms=8, capsize=3, zorder=3)
ro = rows["off (c = 0)"]["r"]
if ro:
    x0 = min(x for x, _ in pts) - 1.2
    ax.errorbar([x0], [ro["delta"]], yerr=[[ro["delta"] - ro["lo"]], [ro["hi"] - ro["delta"]]], color=BLUE,
                marker="s", ms=8, capsize=3, lw=0, elinewidth=2, zorder=3)
    ax.annotate("off (c = 0)", (x0, ro["delta"]), textcoords="offset points", xytext=(8, 0), va="center", color=MUTED)
if SH.get("c_hat"):
    _ref(ax, np.log2(SH["c_hat"]), f"measured fill error\nc = {SH['c_hat']:.2f}", y=0)
ax.set_xlabel("log2 c  (common scale on unread-window mass)")
ax.set_ylabel("Δ LongBench average vs cal (points)")
ax.set_title("End to end: common-scale mass error", loc="left")

ax = axes[0, 1]
ax.axhline(0, color=MUTED, lw=1, zorder=1)
ax.axhspan(-floor, floor, color=BAND, lw=0)
xs, ys, lo, hi = [0], [0.0], [0.0], [0.0]
for m in NOISE_LEVELS:
    r, per = SEEDS[m]
    sd = [p["delta"] for p in per.values() if p]
    ax.scatter([m] * len(sd), sd, s=64, facecolor="none", edgecolor=MUTED, lw=1.5, zorder=2)
    if r:
        xs.append(m); ys.append(r["delta"]); lo.append(r["delta"] - r["lo"]); hi.append(r["hi"] - r["delta"])
ax.errorbar(xs, ys, yerr=[lo, hi], color=BLUE, lw=2, marker="o", ms=8, capsize=3, zorder=3)
ax.annotate("seed mean, 95% CI", (xs[-1], ys[-1]), textcoords="offset points", xytext=(-8, 10), ha="right",
            color=MUTED, fontsize=9)
ax.annotate("open circles: single seeds", (0, 1), xycoords="axes fraction", textcoords="offset points",
            xytext=(4, -4), va="top", color=MUTED, fontsize=9)
ax.set_xticks([0, *NOISE_LEVELS])
ax.set_xlabel(r"$\sigma\,/\,\hat{\sigma}_U$")
ax.set_title("End to end: redistribution, total preserved", loc="left")

if E1C:
    ax = axes[1, 0]
    rs = [("scale0.5", 0.5), ("scale0.8", 0.8), ("cal", 1.0), ("scale1.25", 1.25), ("scale2", 2.0), ("scale4", 4.0)]
    ax.plot([np.log2(c) for _, c in rs], [rung(E1C, r) for r, _ in rs], color=BLUE, lw=2, marker="o", ms=8, zorder=3)
    for r, lab, dy in (("off", "off", 3), ("mass_oracle", "mass oracle", -3)):
        ax.axhline(rung(E1C, r), color=MUTED, lw=1, ls=":", zorder=1)
        ax.annotate(lab, (0, rung(E1C, r)), xycoords=("axes fraction", "data"), textcoords="offset points",
                    xytext=(4, dy), va="bottom" if dy > 0 else "top", color=MUTED, fontsize=9)
    if SH.get("c_hat"):
        _ref(ax, np.log2(SH["c_hat"]), f"c = {SH['c_hat']:.2f}")
    ax.set_xlabel("log2 c")
    ax.set_ylabel("output error vs o_held (median head, median task)")
    ax.set_title("Matched states: common scale", loc="left")
    ax.margins(y=0.12)
    ax = axes[1, 1]
    sg = [0.0, 0.25, 0.5, 1.0, 2.0]
    ax.plot(sg, [rung(E1C, r) for r in ("cal", "noise0.25", "noise0.5", "noise1", "noise2")], color=BLUE, lw=2,
            marker="o", ms=8, zorder=3)
    if SIG:
        for m in NOISE_LEVELS:
            if m * SIG <= 2.2:
                _ref(ax, m * SIG, f"{m}×" + r"$\hat{\sigma}_U$", side="right")
    ax.set_xlabel(r"$\sigma$ (nats)")
    ax.set_title("Matched states: redistribution", loc="left")
fig.text(0.01, -0.02, "Top: 95% bootstrap CIs, paired rows; grey band = largest |Δ| of the floors (cal_repeat, cal_ep). "
         "Bottom: rel. L2 error of the attention output vs reading every held window, same decode states for every "
         "point.", color=MUTED, fontsize=9)
fig.tight_layout()
fig.savefig(FU_RES / "sensitivity_followup.png", dpi=200, bbox_inches="tight")
plt.show()

## 15. Consolidated: attention-output error, accuracy and cost at the fixed KV-memory budget

Every row holds the same KV memory: budget 0.20, `quant_ratio` 0.70 in bytes mode, and the same card widths. The cell checks
this from the sidecars (each row's resolved `retained_bytes` on each task's first example), not from the config. ρ and the
completion term change what is **read** and how it is **combined**. They do not change what is **held**.

| Column | Kind | What it is |
|---|---|---|
| KV held | measured (resolved geometry) | retained K/V bytes, all layers, mean over the four tasks' first examples |
| KV read / step | arithmetic | (held − N_q·(1−ρ)·b_q) / held. Cards are read for every window; int2 data only for the fraction ρ that is opened |
| err vs held | measured, matched states | rel. L2 attention-output error vs reading every held window (Exp 1, median head, median task). It isolates the completion term, and is 0 at ρ = 1.0 by construction |
| err vs full KV | measured, matched states | the same output vs the full-KV model. It adds the eviction and int2 error, which every row shares |
| LongBench | measured | the 4-task average on the arm's own rows (THUDM: OOM = 0) |
| Δ vs cal | measured, paired | on the rows shared with `cal`, with a 95% bootstrap CI. `CI excl. 0` is the significance call |
| s / example | measured, B = 1 | native arms only, on rows 0…N₀−1 (the same rows for every arm), prefill included. Epilogue arms (†) carry an evaluation-only torch epilogue, so their time is not a method cost |

**The summary keeps two claims apart.**
1. Experiment 1 measures how much of the read gate's *internal* approximation error the completion term removes, on identical
   decode states. It is precise, at hundreds of thousands of head-steps per task.
2. The LongBench deltas measure whether that changes *task accuracy*. At this N they resolve only changes larger than the
   printed MDE.

A large, precise (1) next to a non-significant (2) means the downstream effect is smaller than this N can detect. It is **not**
evidence that there is no effect, and it is **not** evidence that (1) carries over to accuracy. The summary is generated from
the numbers, and is written to `followup/results/reviewer2_summary.md` next to the tables.

Decode-time context (`DISTANCE_TO_GOAL.md` §2): the int2 tier is a small share of what a decode step reads, and measured TPOT at
ρ = 1.0 ties with ρ = 0.25 to within 0.2–1.6%. The "KV read" column is a traffic statement, not a latency claim.

In [ ]:
probe = r"""
import json, sys
from modules.quant.quantizer import bytes_per_q_window
from modules.quant.sketch import CARD_BITS_DEFAULT as B, card_field_layout, sketch_bytes_per_head
c = json.load(open(sys.argv[1])); ws = int(sys.argv[2])
H = c["num_key_value_heads"]; D = c.get("head_dim") or c["hidden_size"] // c["num_attention_heads"]
print(json.dumps({"layers": c["num_hidden_layers"], "H_kv": H, "D": D, "ws": ws,
                  "b_qdata": bytes_per_q_window(H, D, ws), "b_card": H * sketch_bytes_per_head(D, ws, B),
                  "b_vbar": H * (card_field_layout(D, B.vm)[0] + 2), "card_bits": B._asdict()}))
"""
# Window prices from the modules that define the formats (per layer, all KV heads), not re-derived here.
GEOM = json.loads(subprocess.check_output([PY, "-c", probe, str(Path(MODEL_DIR) / "config.json"),
                                           str(OPERATING_POINT["window_size"])], cwd=REPO, env=ENV, text=True))

def _geom(n, t):
    s0 = next((s for s in SEGS.get((n, t), []) if s["a"] == 0), None)
    return s0["meta"].get("resolved_geometry_first_example") if s0 else None

GKEYS = ("prefill_len", "retained_bytes", "N_q", "top_k_fp", "bytes_per_gate_card", "total_budget_bytes")
def held_read(names, rho):
    """(KV held in GB over all layers, arithmetic read share per step, same memory as cal on every task?)"""
    held, share, same = [], [], True
    for t in TASKS:
        gs = [g for g in (_geom(n, t) for n in names) if g]
        gc = _geom("cal", t)
        if not gs or not gc:
            return None, None, None
        same &= all(all(g.get(k) == gc.get(k) for k in GKEYS) for g in gs)
        R = gs[0]["retained_bytes"]
        held.append(R * GEOM["layers"] / 1e9)
        share.append((R - gs[0]["N_q"] * (1 - rho) * GEOM["b_qdata"]) / R)
    return float(np.mean(held)), float(np.mean(share)), same

def realised(names):
    v = [s["meta"]["read_gate"]["read_fraction"] for n in names for t in TASKS for s in SEGS.get((n, t), [])]
    return f"{min(v):.3f}" if v and max(v) - min(v) < 5e-4 else (f"{min(v):.3f}-{max(v):.3f}" if v else "—")

def sec_per_ex(names):
    """Measured s/example, mean over tasks, on rows 0..N0-1 (the core rows): native arms only."""
    if any(FU_ARMS[n]["comp"]["mode"] != "native" for n in names):
        return "†"
    v, gpus = [], set()
    for t in TASKS:
        s0 = next((s for s in SEGS.get((names[0], t), []) if s["a"] == 0 and s["b"] == N0[t]), None)
        eps = s0["meta"].get("examples_per_second") if s0 else None
        if not eps:
            return "—"
        v.append(1.0 / eps)
        gpus.add(s0["meta"].get("gpu_name"))
    return f"{np.mean(v):.1f}" + ("" if len(gpus) == 1 else " (mixed GPUs)")

ROWS = [(CEIL, [CEIL], 1.0, "All windows read"),
        ("cal", ["cal"], GATE, "Calibrated completion (QEvict)"),
        ("off", ["off"], GATE, "No completion"),
        ("uncal", ["uncal"], GATE, "Uncalibrated completion"),
        (STRESS[0], [STRESS[0]], STRESS_GATE, "Calibrated completion"),
        (STRESS[1], [STRESS[1]], STRESS_GATE, "No completion"),
        *[(f"scale{c:g}", [f"scale{c:g}"], GATE, f"Calibrated, unread masses x {c:g}") for c in SCALES],
        *[(f"noise{m}x", [f"noise{m}x_s{s}" for s in FU_NOISE_SEEDS], GATE,
           f"Calibrated, redistribution noise {m}x sigma_hat_U") for m in NOISE_LEVELS]]
MS_OF = {CEIL: (0.0, rung(E1C, "full_oracle", "vs full-KV, median head")),
         STRESS[0]: (rung(E1S, "cal"), rung(E1S, "cal", "vs full-KV, median head")),
         STRESS[1]: (rung(E1S, "off"), rung(E1S, "off", "vs full-KV, median head")),
         **{r: (rung(E1C, r), rung(E1C, r, "vs full-KV, median head"))
            for r in ("cal", "off", "uncal", *[f"scale{c:g}" for c in SCALES])},
         **{f"noise{m}x": (noise_rung(None if SIG is None else m * SIG),
                           noise_rung(None if SIG is None else m * SIG, col="vs full-KV, median head"))
            for m in NOISE_LEVELS}}

cons, memory_ok = [], True
for key, names, rho, label in ROWS:
    names = [n for n in names if have(n)]
    if not names:
        continue
    if key.startswith("noise"):
        label += f", mean of {len(names)} seeds" if len(names) > 1 else f", seed {names[0].rsplit('_s', 1)[1]}"
    if key.startswith("noise"):
        dl = SEEDS[int(key[5:-1])][0]
    else:
        dl = None if key == "cal" else delta(key, "cal")
    held, share, same = held_read(names, rho)
    memory_ok &= bool(same)
    e_h, e_f = MS_OF.get(key, (None, None))
    approx = "~" if key.startswith("noise") else ""
    cons.append({"configuration": label, "arm": key if len(names) == 1 else f"{key} ({len(names)} seeds)",
                 "rho": rho_s(rho), "realised rho": realised(names),
                 "KV held (GB)": "—" if held is None else f"{held:.3f}" + ("" if same else " (differs!)"),
                 "KV read/step": "—" if share is None else f"{share:.1%}",
                 "err vs held": "—" if e_h is None else ("0 (by constr.)" if key == CEIL else f"{approx}{e_h:.4f}"),
                 "err vs full KV": "—" if e_f is None else f"{approx}{e_f:.4f}",
                 "LongBench": f"{np.mean([task_score(n, t) for n in names for t in TASKS]):.2f}",
                 "rows/task": min(len(FS[(n, t)][0]) for n in names for t in TASKS),
                 "Δ vs cal [95% CI]": "—" if key == "cal" else ci(dl),
                 "CI excl. 0": "" if dl is None else ("yes" if dl["sig"] else "no"),
                 "MDE80": "" if dl is None else f"{dl['mde80']:.2f}",
                 "s/example": sec_per_ex(names)})
CONS = pd.DataFrame(cons).set_index("configuration")
print(CONS.to_string())
CONS.to_csv(FU_RES / "consolidated.csv")
card_sc = (_geom("cal", TASKS[0]) or {}).get("bytes_per_gate_card")
if card_sc != GEOM["b_card"]:
    print(f"WARNING: the sidecar's card price ({card_sc} B) is not the repo's ({GEOM['b_card']} B) for this model: "
          "the read arithmetic would be off")
print("\nFixed KV memory: " + ("every row holds exactly cal's retained bytes on every task (checked from the sidecars)."
                               if memory_ok else "NOT every row holds cal's bytes: see 'differs!' above."))

# ---------- the Reviewer-2 summary, generated from the numbers above ----------
def _red(E):
    return {t: 1 - E[t]["rungs"]["median head"]["cal"] / E[t]["rungs"]["median head"]["off"] for t in E}
R25, R10 = _red(E1C), _red(E1S)
L = [f"# Reviewer 2: completion accuracy, sensitivity, and the quality-memory trade-off",
     f"Commit `{SHA[:10]}`; Llama-3.1-8B-Instruct; budget {OPERATING_POINT['cache_budget']}, quant_ratio "
     f"{OPERATING_POINT['quant_ratio']} ({OPERATING_POINT['quant_budget_mode']}); LongBench "
     f"{', '.join(TASKS)}. Generated by section 15 of the notebook.", ""]
L += ["## 1. Internal approximation error (Experiment 1: measured, matched decode states)"]
if R25:
    cands = {"median of per-task reductions": float(np.median(list(R25.values()))),
             "mean of per-task reductions": float(np.mean(list(R25.values()))),
             "1 - median_task err(cal) / median_task err(off)": 1 - rung(E1C, "cal") / rung(E1C, "off")}
    hit = [k for k, v in cands.items() if abs(v - PAPER_ERROR_REDUCTION) <= 0.01]
    L += [f"* At rho = {rho_s(GATE)}, calibrated completion removes **{cands['median of per-task reductions']:.0%}** of the "
          f"read gate's attention-output error (median over tasks; per task "
          + ", ".join(f"{t} {v:.0%}" for t, v in R25.items())
          + f"; {sum(E1C[t]['head_steps'] for t in E1C):,} head-steps). Error without completion "
          f"{rung(E1C, 'off'):.4f}, with it {rung(E1C, 'cal'):.4f}; a perfect mass estimate would reach "
          f"{rung(E1C, 'mass_oracle'):.4f}.",
          f"* Check against the quoted {PAPER_ERROR_REDUCTION:.0%}: "
          + (f"reproduced as the {hit[0]}." if hit else "NOT reproduced by any of "
             + ", ".join(f"{k} = {v:.1%}" for k, v in cands.items()) + ". Re-check the quoted figure before sending.")]
if R10:
    L += [f"* At rho = {rho_s(STRESS_GATE)} it removes **{np.median(list(R10.values())):.0%}** (per task "
          + ", ".join(f"{t} {v:.0%}" for t, v in R10.items()) + ")."]
L += ["* This is a property of the approximation on identical states. Its precision does not depend on the LongBench N.", ""]

L += ["## 2. Downstream accuracy (LongBench: measured, paired, 95% bootstrap CI)"]
if B25:
    n_t = B25["n"] // len(TASKS)
    L += [f"* Completion vs none at rho = {rho_s(GATE)}: cal - off = **{ci(B25)}** points on the 4-task average, ~{n_t} "
          f"rows/task. " + ("The CI includes 0: **not statistically significant** at this N." if not B25["sig"] else
                            "The CI excludes 0: significant at this N.")
          + f" The smallest change this N detects with 80% power is {B25['mde80']:.2f} points."]
    if R25 and not B25["sig"]:
        L += [f"* Read together: the internal error reduction is large and precisely measured, and the downstream "
              f"difference is below what {n_t} rows/task can resolve. The data support *completion makes the "
              f"approximation substantially more accurate*. They do not yet support *completion changes LongBench "
              f"accuracy*, and they do not show that it doesn't."]
    elif R25:
        L += [f"* Read together: the internal error reduction and a significant downstream difference "
              + ("point the same way: completion improves LongBench accuracy at this N." if B25["delta"] > 0 else
                 "point opposite ways: completion LOWERS LongBench accuracy at this N despite the smaller internal "
                 "error. Report this as found; it needs explaining before the claim is made.")]
L += [""]

L += [f"## 3. rho = 1.00 (every window read) vs calibrated completion at rho = {rho_s(GATE)}"]
short = [t for t in TASKS if (CEIL, t) not in FS]
L += [f"* Availability: " + ("all four tasks verified (gate verdict 'gated', realised read fraction "
                              f"{realised([CEIL])})." if not short else f"missing on {short}.")]
r1 = delta("cal", CEIL)
if r1:
    L += [f"* cal@{rho_s(GATE)} - rho 1.00 = {ci(r1)} points (" + ("CI excludes 0" if r1["sig"] else "CI includes 0") + ")."]
    gap = ratio_stats((CEIL, "cal", "off"), lambda A: A["cal"] - A["off"], lambda A: A[CEIL] - A["off"], ("gap",))
    L += ["* Share of the read-gate gap closed by completion: " + (
        f"{gap['point']:.2f} [{gap['lo']:.2f}, {gap['hi']:.2f}]." if gap and gap["den_lo"] > 0 else
        "undefined: rho = 1.00 does not beat 'off' significantly at this N, so there is no resolvable gap to close.")]
L += [""]

L += [f"## 4. rho = {rho_s(STRESS_GATE)} stress test"]
if len(MED) == 2:
    a, b = MED[GATE], MED[STRESS_GATE]
    L += [f"* Matched states: the unread share of held attention goes {a['unread']:.1%} -> {b['unread']:.1%}; output "
          f"error without completion {a['err_off']:.4f} -> {b['err_off']:.4f}, with it {a['err_cal']:.4f} -> "
          f"{b['err_cal']:.4f}. The error completion removes goes {a['removed_abs']:.4f} -> {b['removed_abs']:.4f} "
          f"({b['removed_abs'] / a['removed_abs']:.2f}x), or {a['removed']:.0%} -> {b['removed']:.0%} of it."]
if B10:
    L += [f"* End to end: cal - off = {ci(B10)} at rho = {rho_s(STRESS_GATE)} vs {ci(B25)} at rho = {rho_s(GATE)}; "
          f"difference-in-differences {ci(DID)} ("
          + ("CI excludes 0" if DID and DID["sig"] else "CI includes 0: this N cannot tell") + ")."]
L += [""]

L += ["## 5. End-to-end sensitivity (Δ vs cal, points [95% CI])"]
for c in SCALES:
    r = rows[f"scale c = {c:g}"]["r"]
    if r:
        L += [f"* masses x {c:g}: {ci(r)}" + (" *" if r["sig"] else "")]
for m in NOISE_LEVELS:
    r, per = SEEDS[m]
    if r:
        seeds = ", ".join(f"{p['delta']:+.2f}" for p in per.values() if p)
        L += [f"* noise {m}x sigma_hat_U, " + (f"mean of {len(per)} seeds" if len(per) > 1 else "one seed")
              + f": {ci(r)}" + (" *" if r["sig"] else "")
              + (f" (single seeds {seeds})" if len(per) > 1 else "")]
L += [f"* {n_sig} of {len(sens_rows)} points have a CI excluding 0 (marked *). Unadjusted: about "
      f"{0.05 * len(sens_rows):.1f} would by chance. Floors (|Δ| of a repeat / the epilogue at identity): "
      + ", ".join(f"{k[7:]} {rows[k]['r']['delta']:+.2f}" for k in ("floor: cal_repeat", "floor: cal_ep")
                  if rows[k]["r"]) + ".", ""]

L += ["## 6. Cost at the fixed KV-memory budget"]
held_c = held_read(["cal"], GATE)[0]
L += [f"* Every row holds the same KV memory ("
      + ("?" if held_c is None else f"{held_c:.3f}") + f" GB over {GEOM['layers']} layers, mean of the tasks' first "
      f"examples)" + ("; checked from the sidecars." if memory_ok else "; NOT confirmed by the sidecars.")]
for key, rho in ((CEIL, 1.0), ("cal", GATE), (STRESS[0], STRESS_GATE)):
    if have(key) and held_read([key], rho)[1] is not None:
        L += [f"* rho = {rho_s(rho)}: reads {held_read([key], rho)[1]:.1%} of the held bytes per decode step (arithmetic); "
              f"measured {sec_per_ex([key])} s/example at B = 1 (rows 0..N0-1, prefill included)."]
L += [f"* The completion term's only resident cost is the value centroid on each card: {GEOM['b_vbar']} B per int2 "
      f"window across {GEOM['H_kv']} KV heads, {GEOM['b_vbar'] / (GEOM['b_qdata'] + GEOM['b_card']):.1%} of the "
      f"{GEOM['b_qdata'] + GEOM['b_card']} B the window costs. 'off' as run here holds it anyway (it is the same cache with "
      f"the term removed in an epilogue), so the rows above hold equal memory.",
      "* Decode time barely tracks these bytes (DISTANCE_TO_GOAL.md §2-3): the read column is traffic, not latency.", ""]
L += ["## Caveats",
      f"* Experiment 1's prompts are the first {EXP1_PREFILL} tokens of each task's documents plus a greedy "
      "continuation, not the templated task prompt.",
      "* The LongBench rows are the first N of each task file; every CI is a paired bootstrap over rows, with tasks "
      "resampled independently. They are not corrected for multiple comparisons."]
SUMMARY = "\n".join(L)
(FU_RES / "reviewer2_summary.md").write_text(SUMMARY + "\n")

md = ["| " + " | ".join(["Configuration", *CONS.columns]) + " |", "|---" * (len(CONS.columns) + 1) + "|"]
md += ["| " + " | ".join([str(i), *[str(v) for v in r]]) + " |" for i, r in CONS.iterrows()]
(FU_RES / "consolidated.md").write_text("\n".join(md) + f"\n\ncommit {SHA}\n")
print("\n" + SUMMARY)
print(f"\nwritten: {FU_RES}")